# Alaska Region Data Management Plan Script

## Description
This Jupyter Notebook utilizes Python to scrape information from the Alaska Region Data Management Plan template (https://doi.org/10.7944/P9XUYMQT). Relevant metadata attriutes are then traslated into the mdJSON metadata standard and written to an mdEditor file containing both project and product metadata records.

This notebook can also be used to populate an Excel spreadsheet template that can by imported into the National Data Management Plan SharePoint list, using a PowerAutomate Flow developed by T. Wisneskie (https://ecos.fws.gov/ServCat/Reference/Profile/148873).

## Disclaimer
The quality of metadata is the responsibility of the metadata authors and data steward(s) described in the data management plan. The initiated metadata file is only intended to be a starting point. This file should be updated by metadata authors as the project progresses, and additional records such as data dictionaries may need to be added.

## Usage
The first block of code below should be updated to point to the relevant files and directories (DMP, blank DMP template with the same version, metadata template directory, and metadata contact directory). Settings can also be updated to reflect user preferences on whether or not quality assurance/control information, a data disclaimer, and custom profiles/schemas should be included in the mdEditor metadata. If custom profiles/schemas are desired, supply the pathways to the raw profiles and schemas on GitHub. No further user input is required unless further customization is desired.

### Additional Information
Last updated: 08/25/2025
Contact: caylen_cummins@fws.gov

In [ ]:
# ================================================================
# SETTINGS -- this replaces the old 'dmp'/'blankdmp' docx pathways.
# ================================================================

# Pathway to the DMP data file exported from DMP_Interface.html
# (the "Export DMP data (.json)" button on the Review & Export screen).
# This replaces the old 'dmp' Word-document pathway.
dmp_json_path = r'C:\Users\tpatterson\Downloads\walrus_haulout_monitoring.json'

# Pathway to the folder containing dmp_json_to_dataframe.py
# (the bridge script that ships alongside DMP_Interface.html).
bridge_dir = r'C:\Users\tpatterson\Downloads\AK_DMP_Interface'

# Pathway to the mdEditor contacts export containing the FWS Region and
# FWS Program organization records (used to add them as 'administrator'
# contacts -- see region_program_contacts.py).
region_program_contacts_path = r'C:\Users\tpatterson\Downloads\AK_DMP_Interface\FWSRegion_Program_Contacts_mdeditor-20260811-235138.json'

# Pathway to the contacts DIRECTORY you want to use to check against existing vs. new contacts;
# The script will later search for the most recent file within the directory, provided ISO datetime is in the filename
contact_folder = r'C:\Users\tpatterson\OneDrive - DOI\Documents\DM_Metadatafiles\AK_contacts_profiles'

# Pathway to directory that holds mdJSON/mdEditor template txt files
tmp_dir = r'C:\Users\tpatterson\OneDrive - DOI\Documents\GitHub\DMPythonScript\ak_dmp_script-main-march2024\ak_dmp_script-main\mdJSON_templates'

#Pathway to the Excel file for uploading DMPs to the HQ SharePoint list
# Note: this file MUST be on One Drive to work with Power Automate and to version in case of errors in the script
dmp_spreadsheet = r'C:\Users\tpatterson\OneDrive - DOI\DMP_Import\National_DMPs.xlsx'

# True or False, do you want to include QA/QC descriptions in the lineage section of these records?
incl_quality = True

# True or False, do you want to include the following data disclaimer for open products?
# Although these data have been processed successfully on a computer system at the U.S. Fish and Wildlife Service, 
# no warranty expressed or implied is made regarding the display or utility of the data for other purposes, 
# nor on all computer systems, nor shall the act of distribution constitute any such warranty.
incl_disclaimer = True

# True or False, do you want to import profiles and schemas and associate them with records?
import_prof_schema = True

# URL to current mdEditor PROJECT PROFILE for your program; input False above if no profile is needed and ignore the lines below
proj_profile = r'https://raw.githubusercontent.com/USFWS/ak-md-profiles/main/ak-proj-profile.json'

# URL to current mdEditor PRODUCT PROFILE for your program
prod_profile = r'https://raw.githubusercontent.com/USFWS/ak-md-profiles/main/ak-prod-profile.json'

# URL to current mdEditor PROJECT SCHEMA for your program
proj_schema = r'https://raw.githubusercontent.com/USFWS/ak-md-profiles/main/ak-proj-schema.json'

# URL to current mdEditor PRODUCT SCHEMA for your program
prod_schema = r'https://raw.githubusercontent.com/USFWS/ak-md-profiles/main/ak-prod-schema.json'

# URL to current mdEditor DICTIONARY PROFILE for your program
dict_profile = r'https://raw.githubusercontent.com/USFWS/ak-md-profiles/main/ak-data-dict-profile.json'

# URL to current mdEditor DICTIONARY SCHEMA for your program
dict_schema = r'https://raw.githubusercontent.com/USFWS/ak-md-profiles/main/ak-data-dict-schema.json'


***
# Preparing DMP information
This first section of the script loads information from the data management plan. You must run these blocks of code for both initiating mdEditor metadata and for pushing data to the national DMP list.

### Loading basic required info
The following blocks of code load the basic requirements to run the script: libraries and the most recent mdEditor contact file in the specified directory.

**Note:** This notebook now reads the DMP directly from the JSON file exported by `DMP_Interface.html`, instead of scraping a filled-out Word document. The `dmp_json_to_dataframe.py` bridge script (in `bridge_dir`, set above) rebuilds the exact same `df` dataframe the rest of this notebook expects, so everything from here on is unchanged from the original Word-based workflow.

In [ ]:
#specific to extracting information from word documents
import os
import zipfile
#other tools useful in extracting the information from our document
import re

# Library for reading online JSON profiles/schema
import urllib.request

#data frame library
import pandas as pd
#library for reading json files
import json
from pandas import json_normalize
#library to generate uids
import uuid
# numpy library
import numpy as np
# datetimes
from datetime import datetime, date
import string

#supressing warnings about deprecated functions for ease of visual display
import warnings

def fxn():
    warnings.warn("deprecated", DeprecationWarning)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    fxn()
# Bridge module: converts the DMP_Interface.html JSON export into the same
# 'df' dataframe shape this notebook previously got from scraping a docx.
import sys
sys.path.append(bridge_dir)
import dmp_json_to_dataframe as bridge
import region_program_contacts as rpc


In [3]:
# Search for most recent contact file within the contact directory specified in the first block

contact_files = []
for filename in os.listdir(contact_folder):
    f = os.path.join(contact_folder, filename)
    if os.path.isfile(f) and '.json' in f:
        contact_files.append(f)
    
contact_files = pd.DataFrame(contact_files, columns=['fn'])
# Pulling out datetime from filename, then sorting by datetime descending
contact_files['datetime'] = contact_files['fn'].str.extract(r"(\d{4}[01]\d[0-3]\d.*)")
contact_files = contact_files.sort_values('datetime', ascending=False).reset_index(drop=True)

# Selecting the most recent filename
contact_md = contact_files['fn'][0]

print('The following contact file will be used:')
display(contact_md)

The following contact file will be used:


'C:\\Users\\tpatterson\\OneDrive - DOI\\Documents\\DM_Metadatafiles\\AK_contacts_profiles\\AK-Contacts-mdeditor-20260721-001521.json'

### Loading the completed data management plan
The block of code below loads the DMP data exported from `DMP_Interface.html` and builds the same `df` dataframe (columns: `field`, `value`, `prod_num`, `sample_num`) that the original notebook produced by scraping a filled-out Word document. Every cell below this point is unchanged from the original notebook.

In [ ]:
# Building df, dmp, and dmpvers from the interface's JSON export
# (replaces the old 'read blank docx' + 'read completed docx' cells)
df, dmp, dmpvers = bridge.build_dataframe(dmp_json_path)

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
display(df)


### Pulling contacts
The next several blocks of code prepare contact information. Contact emails listed in the DMP are compared against the specified mdEditor contact file. If the contact exists, that contact information is pulled and will be added to the initiated mdEditor metadata file. If the contact does not exist, a new contact will be created. Be sure to add any new contacts to the appropriate programmatic and regional contact lists to prevent duplicates in the future.

In [6]:
# Extracting contact columns into a separate data frame
contacts = df[(df['field'].str.startswith('contact'))|(df['field'].str.startswith('productMetaAuthor'))|
             (df['field'].str.startswith('productOriginator'))|(df['field'].str.startswith('dmpDM'))|
              (df['field'].str.startswith('dmpCreator'))|(df['field'].str.startswith('dmpUpdater'))]

contacts = contacts.drop(columns=['prod_num','sample_num'])

# Assign a type of contact, pulled from 'field' value
for i, row in contacts.iterrows():
    ctype = row['field']
    string = re.findall("Primary|PI|Steward|Custodian|Trustee|Other|Author|Originator|DM|Creator|Updater", ctype)[0]
    if re.search('[0-9]{1,}-[0-9]{1,}$',ctype):
        num = re.findall("[0-9]{1,}-[0-9]{1,}", ctype)[0]
        contacts.at[i,'type'] = string+str(num)
    elif re.search('[0-9]{1,}$',ctype):
        num = re.findall("[0-9]{1,}", ctype)[0]
        contacts.at[i,'type'] = string+str(num)
    else:
        contacts.at[i,'type'] = string

# Assigning a column name for when the data are transposed
for i, row in contacts.iterrows():
    ctype = row['type']
    if re.search('[0-9]{1,}-[0-9]{1,}$',ctype):
        num = re.findall("[0-9]{1,}-[0-9]{1,}$", ctype)[0]
        string = row['field'].replace('contact','').replace(ctype,'').replace('product','').replace(num,'').replace(
            'Other','').replace('Meta','').replace('Originator','').replace('Author','').replace('dmp','')
    elif re.search('[0-9]{1,}$',ctype):
        num = re.findall("[0-9]{1,}", ctype)[0]
        string = row['field'].replace('contact','').replace(ctype,'').replace('product','').replace(num,'').replace(
            'Other','').replace('Meta','').replace('Originator','').replace('Author','').replace('dmp','')
    else:
        string = row['field'].replace('contact','').replace(ctype,'').replace('product','').replace('Other','').replace(
            'Meta','').replace('Originator','').replace('Author','').replace('dmp','')
    contacts.at[i,'column'] = string
    
# Assigning a number to each contact
contacts['contact_num'] = contacts.groupby('type').ngroup().add(1)

# Reformatting contacts list to field and value columns, with contact number associated
contacttypes = contacts[['type','contact_num']].drop_duplicates().rename(columns={'type':'value'})
contacttypes['column'] = 'contact_type'
contacts = pd.concat([contacts.drop(columns=['type','field']), contacttypes]).sort_values(['contact_num','column'])
contacts['field'] = contacts['column']+contacts['contact_num'].astype(str)
    
# Getting a unique list of contact numbers
contactcount = contacts['contact_num'].values.tolist()
contactcount = list(set(contactcount))
contactcount = contactcount

# Transposing rows to columns
contacts_t =  contacts.drop(columns=['contact_num','column']).set_index('field').transpose().reset_index(drop=True)

display(contacts_t)

field,Email1,FirstName1,LastName1,Org1,contact_type1,Email2,FirstName2,LastName2,Org2,contact_type2,Email3,FirstName3,LastName3,contact_type3,Email4,FirstName4,LastName4,NonDOI4,Org4,Phone4,Position4,contact_type4,Email5,FirstName5,LastName5,contact_type5,Email6,FirstName6,LastName6,NonDOI6,Org6,contact_type6,Email7,FirstName7,LastName7,NonDOI7,Org7,contact_type7,Email8,FirstName8,LastName8,NonDOI8,Org8,contact_type8,Email9,FirstName9,LastName9,NonDOI9,Org9,contact_type9,Email10,FirstName10,LastName10,Org10,Phone10,Position10,Role10,contact_type10,Email11,FirstName11,LastName11,Org11,Phone11,Position11,contact_type11,Email12,FirstName12,LastName12,Org12,Phone12,Position12,contact_type12,Email13,FirstName13,LastName13,Phone13,Position13,contact_type13,Email14,FirstName14,LastName14,contact_type14
0,Devin_johnson@fws.gov,Devin,Johnson,,Author1-1,Devin_johnson@fws.gov,Devin,Johnson,,Author2-1,Devin_johnson@fws.gov,Devin,Johnson,Creator,Devin_johnson@fws.gov,Devin,Johnson,False,,,Wildlife Biologist,Custodian,Tamatha_patterson@fws.gov,Tamatha,Patterson,DM,Devin_johnson@fws.gov,Devin,Johnson,False,,Originator1-1,Devin_johnson@fws.gov,Devin,Johnson,False,,Originator2-1,Irina_trukhanova@fws.gov,Irina,Trukhanova,False,,Originator2-2,Sarah_hoepfner@fws.gov,Sarah,Hoepfner,False,,Originator2-3,Jonah_withers@fws.gov,Jonah,Withers,,,,custodian,Other,Devin_johnson@fws.gov,Devin,Johnson,,,Wildlife Biologist,Primary,Devin_johnson@fws.gov,Devin,Johnson,,,Wildlife Biologist,Steward,Daniel_bjornlie@fws.gov,Daniel,Bjornlie,,,Trustee,Devin_johnson@fws.gov,Devin,Johnson,Updater


In [7]:
# Combining product records into a single data frame, where each row represents a seperate contact
contacts_df = pd.DataFrame()

for i in contactcount:
    re = '[a-zA-Z]'+str(i)+'$'
    sub = contacts_t.filter(regex=re).dropna(how='all')
    sub.columns = sub.columns.str.replace("[0-9]+", "", regex=True)
    contacts_df = pd.concat([contacts_df,sub])
    
contacts_df = contacts_df.reset_index(drop=True).fillna('')

# Removing primary contact, who is just the point of contact for the DM in regards to the DMP
contacts_df = contacts_df[~(contacts_df.contact_type=='Primary')]

# Removing empty contacts created by repeating content controls
contacts_df = contacts_df[~((contacts_df.Email=='')&(contacts_df.FirstName=='')&(contacts_df.LastName==''))]

# Moving product number info into a separate field and removing it from contact_type, for authors and originators
contacts_df['product_num'] = contacts_df['contact_type'].str.extract('([0-9]{0,}-?[0-9]{1,})')
contacts_df['product_num'] = contacts_df['product_num'].str.replace('-[0-9]{1,}','',regex=True)
contacts_df['contact_type'] = contacts_df['contact_type'].str.replace('([0-9]{0,}-?[0-9]{1,})','',regex=True)
# Removing product number from project-related other contacts
contacts_df.loc[~((contacts_df.contact_type=='Author')|(contacts_df.contact_type=='Originator')),'product_num'] = ''
contacts_df.loc[(contacts_df.product_num.isna()),'product_num'] = ''

contacts_df.loc[(contacts_df.contact_type=='Author'),'Role'] = 'author'
contacts_df.loc[(contacts_df.contact_type=='Custodian'),'Role'] = 'custodian'
contacts_df.loc[(contacts_df.contact_type=='PI'),'Role'] = 'principalInvestigator'
contacts_df.loc[(contacts_df.contact_type=='Primary'),'Role'] = 'pointOfContact'
contacts_df.loc[(contacts_df.contact_type=='Steward'),'Role'] = 'pointOfContact'
contacts_df.loc[(contacts_df.contact_type=='Trustee'),'Role'] = 'owner'
contacts_df.loc[(contacts_df.contact_type=='Originator'),'Role'] = 'originator'
contacts_df.loc[(contacts_df.contact_type=='DM'),'Role'] = 'custodian'
contacts_df.loc[(contacts_df.contact_type=='Creator'),'Role'] = 'originator'
contacts_df.loc[(contacts_df.contact_type=='Updater'),'Role'] = 'originator'

contacts_df['Email'] = contacts_df['Email'].str.lower().str.strip()

# Removing Other contact, if empty
contacts_df = contacts_df[~((contacts_df.contact_type=='Other')&(contacts_df.Email=='')
                           &(contacts_df.FirstName=='')&(contacts_df.LastName=='')&(contacts_df.NonDOI=='')
                           &(contacts_df.Org=='')&(contacts_df.Phone=='')&(contacts_df.Position==''))]
# Removing Updater contact, if empty
contacts_df = contacts_df[~((contacts_df.contact_type=='Updater')&(contacts_df.Email=='')
                           &(contacts_df.FirstName=='')&(contacts_df.LastName=='')&(contacts_df.NonDOI=='')
                           &(contacts_df.Org=='')&(contacts_df.Phone=='')&(contacts_df.Position==''))]

# Adding contacts for the DMP metadata record
newprodnum = str(max(contacts_df[(contacts_df.product_num!='')]['product_num'].astype(int))+1)
dmrow = contacts_df[(contacts_df.contact_type=='Creator')|
            (contacts_df.contact_type=='Updater')]
dmrow['product_num'] = newprodnum
dmrow['Role'] = 'author'
contacts_df = pd.concat([contacts_df,dmrow])
contacts_df.loc[(contacts_df.contact_type=='Creator')|
            (contacts_df.contact_type=='Updater'),'product_num'] = newprodnum

contacts_df = contacts_df.reset_index(drop=True)

#Removing duplicates of a contact fulfilling same role for the same product
contacts_df = contacts_df.drop_duplicates(
    subset = ['Email', 'Role','product_num'],
    keep = 'first').reset_index(drop = True)

#Removing contact roles that are tbd
contacts_df = contacts_df[~((contacts_df.FirstName=='TBD')&(contacts_df.LastName=='TBD'))]

display(contacts_df)

C:\Users\tpatterson\AppData\Local\Temp\1\ipykernel_27908\3894257295.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dmrow['product_num'] = newprodnum
C:\Users\tpatterson\AppData\Local\Temp\1\ipykernel_27908\3894257295.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dmrow['Role'] = 'author'


field,Email,FirstName,LastName,Org,contact_type,NonDOI,Phone,Position,Role,product_num
0,devin_johnson@fws.gov,Devin,Johnson,,Author,,,,author,1
1,devin_johnson@fws.gov,Devin,Johnson,,Author,,,,author,2
2,devin_johnson@fws.gov,Devin,Johnson,,Creator,,,,originator,3
3,devin_johnson@fws.gov,Devin,Johnson,,Custodian,False,,Wildlife Biologist,custodian,
4,tamatha_patterson@fws.gov,Tamatha,Patterson,,DM,,,,custodian,
5,devin_johnson@fws.gov,Devin,Johnson,,Originator,False,,,originator,1
6,devin_johnson@fws.gov,Devin,Johnson,,Originator,False,,,originator,2
7,irina_trukhanova@fws.gov,Irina,Trukhanova,,Originator,False,,,originator,2
8,sarah_hoepfner@fws.gov,Sarah,Hoepfner,,Originator,False,,,originator,2
9,jonah_withers@fws.gov,Jonah,Withers,,Other,,,,custodian,


In [8]:
# Reading in existing contacts from master file to compare with contacts listed in DMP
with open(contact_md, 'r') as handle:
    parsed = json.load(handle)
existing_contacts = pd.DataFrame.from_dict(parsed, orient="index").transpose().reset_index(drop=True)
existing_contacts['data'] = existing_contacts['data'].apply(json.dumps)
existing_contacts = existing_contacts.rename(columns={'data':'contact_json'})

display(existing_contacts.head())

,contact_json
0,"{""id"": ""t5ebaijs"", ""attributes"": {""json"": ""{\""contactId\"":\""5236f540-d67b-4e41-b533-598258ae93b3\"",\""isOrganization\"":true,\""name\"":\"" American Meteorological Society\"",\""memberOfOrganization\"":[],\""logoGraphic\"":[],\""phone\"":[],\""address\"":[{\""addressType\"":[\""mailing\""],\""description\"":\"" American Meteorological Society\""}],\""electronicMailAddress\"":[],\""onlineResource\"":[],\""hoursOfService\"":[],\""externalIdentifier\"":[]}"", ""date-updated"": ""2019-03-29T14:07:58.934Z"", ""rev"": null}, ""type"": ""contacts"", ""meta"": {""title"": "" American Meteorological Society"", ""icon"": ""users"", ""export"": true}}"
1,"{""id"": ""9rdo1n95"", ""attributes"": {""json"": ""{\""contactId\"":\""0e26cc9a-2f9b-41e8-b714-3006793c78fa\"",\""isOrganization\"":true,\""name\"":\"" Utqia\u0121vik (Barrow) Field Office\"",\""memberOfOrganization\"":[\""821858df-5d0e-445a-b027-014f5ef68782\""],\""logoGraphic\"":[],\""phone\"":[{\""service\"":[\""voice\""],\""phoneName\"":\""Main Office\"",\""phoneNumber\"":\""(907) 852-2058\""}],\""address\"":[{\""addressType\"":[\""physical\"",\""mailing\""],\""deliveryPoint\"":[\""5146 Boxer St\""],\""city\"":\""Utqia\u0121vik\"",\""postalCode\"":\""99723\"",\""administrativeArea\"":\""AK\"",\""country\"":\""USA\""}],\""electronicMailAddress\"":[],\""onlineResource\"":[],\""hoursOfService\"":[],\""contactType\"":\""federal\"",\""externalIdentifier\"":[]}"", ""date-updated"": ""2021-04-29T16:31:17.813Z"", ""rev"": null}, ""type"": ""contacts"", ""meta"": {""title"": "" Utqia\u0121vik (Barrow) Field Office"", ""icon"": ""users"", ""export"": true}}"
2,"{""id"": ""9nblejdq"", ""attributes"": {""json"": ""{\""contactId\"":\""60e8e211-2673-406d-b459-10aa40be3956\"",\""isOrganization\"":true,\""name\"":\""ABR inc.\"",\""memberOfOrganization\"":[],\""logoGraphic\"":[{\""fileUri\"":[{\""uri\"":\""https://breadlineak.org/wp-content/uploads/abrlogo-green.png\"",\""name\"":\""ARBinc\""}],\""fileName\"":\""ARB inc.\""}],\""phone\"":[{\""service\"":[\""voice\""],\""phoneName\"":\""Fairbanks Office\"",\""phoneNumber\"":\""907-455-6777\""},{\""service\"":[\""voice\""],\""phoneName\"":\""Anchorage Office\"",\""phoneNumber\"":\""907-344-6777\""}],\""address\"":[{\""addressType\"":[\""mailing\""],\""deliveryPoint\"":[\""PO Box 80410\""],\""city\"":\""Fairbanks\"",\""administrativeArea\"":\""Alaska\"",\""postalCode\"":\""99708\"",\""country\"":\""USA\""},{\""addressType\"":[\""mailing\""],\""deliveryPoint\"":[\""PO Box 240268\""],\""city\"":\""Anchorage\"",\""administrativeArea\"":\""Alaska\"",\""postalCode\"":\""99524\"",\""country\"":\""USA\""}],\""electronicMailAddress\"":[\""info@abrinc.com\""],\""onlineResource\"":[{\""name\"":\""ABR Inc | Environmental Consulting Services\"",\""uri\"":\""https://www.abrinc.com/\"",\""function\"":\""information\""}],\""hoursOfService\"":[],\""contactType\"":\""research\"",\""externalIdentifier\"":[]}"", ""date-updated"": ""2021-08-19T00:51:25.560Z"", ""rev"": null}, ""type"": ""contacts"", ""meta"": {""title"": ""ABR inc."", ""icon"": ""users"", ""export"": true}}"
3,"{""id"": ""ldtkemam"", ""attributes"": {""json"": ""{\""contactId\"":\""6195fafb-d2d3-44ef-bf30-62f7fdef4a7c\"",\""isOrganization\"":false,\""name\"":\""Aaron Martin\"",\""memberOfOrganization\"":[\""d4060e2e-d9eb-4972-a85f-32d3e9b244b9\""],\""logoGraphic\"":[],\""phone\"":[{\""service\"":[\""voice\""],\""phoneNumber\"":\""907-786-3510\"",\""phoneName\"":\""Phone\""}],\""address\"":[],\""electronicMailAddress\"":[\""aaron_e_martin@fws.gov\""],\""onlineResource\"":[],\""hoursOfService\"":[],\""contactType\"":\""federal\"",\""positionName\"":\""Invasive Species Program Coordinator\"",\""externalIdentifier\"":[]}"", ""date-updated"": ""2021-02-03T01:00:57.033Z"", ""rev"": null}, ""type"": ""contacts"", ""meta"": {""title"": ""Aaron Martin"", ""icon"": ""users"", ""export"": true}}"
4,"{""id"": ""e7fhi1um"", ""attributes"": {""json"": ""{\""contactId\"":\""6a52baf0-9603-4

In [9]:
pd.set_option('display.max_colwidth', 200)

# Comparing existing list with people listed in DMP
        
final_contacts = contacts_df.copy()

for i, row in existing_contacts.iterrows():
    jsonstr = row['contact_json']
    for j, row in final_contacts.iterrows():
        cemail = str(row['Email'])
        if cemail!='' and cemail in jsonstr.lower():
            final_contacts.at[j,'json'] = jsonstr
            
# Getting contact uuid
final_contacts['uuid'] = final_contacts['json'].str[60:96]

# Adding the project's FWS Region and FWS Program(s) as contacts, based on
# the selections made in DMP_Interface.html. Region is added as
# administrator/distributor/publisher; each Program is added as
# administrator. (Replaces the old hardcoded "AK Region USFWS" block --
# see region_program_contacts.py.)
admrows = rpc.build_admin_rows(df, region_program_contacts_path)
final_contacts = pd.concat([final_contacts,admrows]).reset_index(drop=True)
cols = final_contacts.columns[:-2]
final_contacts[cols] = final_contacts[cols].fillna('') #Filling np.nan with string for all but json and uuid columns

for i, row in existing_contacts.iterrows():
    jsonstr = row['contact_json']
    for j, row in final_contacts.iterrows():
        if type(row['json'])==float:
            cuid = str(row['uuid'])
            if cuid!='' and 'contactId\\\":\\\"'+cuid in jsonstr:
                final_contacts.at[j,'json'] = jsonstr

# Getting organization uuid, by searching for uuids and removing the first in the list (contact's id)
final_contacts['org_uuid'] = final_contacts['json'].str.findall(
            '([a-fA-F0-9]{8}-[a-fA-F0-9]{4}-[a-fA-F0-9]{4}-[a-fA-F0-9]{4}-[a-fA-F0-9]{12})')
# Removing first uuid from each list, since that uuid is for the contact itself, not an organization
for i, row in final_contacts.iterrows():
    if type(row['org_uuid'])!=float:
        final_contacts.at[i,'org_uuid'] = row['org_uuid'][1:]        
            
# Assigning uuid to each new contact
newc = final_contacts[(final_contacts.uuid.isna())][['Email']].drop_duplicates().reset_index(drop=True)
newc['uuid'] = newc.apply(lambda x: uuid.uuid4(), axis=1)

for i, row in newc.iterrows():
    nuid = row['uuid']
    nemail = row['Email']
    for j, row in final_contacts.iterrows():
        if nemail in row['Email']:
            final_contacts.at[j,'uuid'] = nuid
            
# Checking new contacts for existing organizations and finding the org uuid based on the name
for j, row in final_contacts.iterrows():
    if type(row['org_uuid'])==float and row['Org']!='':
        norg = str(row['Org'])
        norgj = ',\\\"name\\\":\\\"'+norg+'\\\"'
        for i, row in existing_contacts.iterrows():
            jsonstr = row['contact_json']
            if norgj in jsonstr:
                final_contacts.at[j,'org_uuid'] = jsonstr[60:96]
  
            
# Assigning uuid to each new organization
norg = final_contacts[(final_contacts.org_uuid.isna())&(final_contacts.Org!='')
                     ][['Org']].drop_duplicates().reset_index(drop=True)
norg['uuid'] = norg.apply(lambda x: uuid.uuid4(), axis=1)

for i, row in norg.iterrows():
    nuid = row['uuid']
    norgname = row['Org']
    for j, row in final_contacts.iterrows():
        if norgname in row['Org']:
            final_contacts.at[j,'org_uuid'] = nuid
            
# Preparing template for new contacts
with open(os.path.join(tmp_dir,'mdEd_contact.txt'), 'r') as handle:
    contact_txt = handle.read()
    handle.close()
with open(os.path.join(tmp_dir,'individual_contact.txt'), 'r') as handle:
    ind_txt = handle.read()
    handle.close()
    
# Filling out json for empty values
for i, row in final_contacts.iterrows():
    if type(row['json'])==float:
        # Generating an 8 digit ID for the record
        contact_id = str(uuid.uuid4()).lower()[:8]
        # Getting current datettime
        currentdatetime = str(datetime.now())
        currentdatetime = currentdatetime[:-3]
        currentdatetime = currentdatetime.replace(' ','T')+'Z'
        
        # Getting position string
        if row['Position'] == '':
            postr = ''
        else:
            postr = ',\\\"positionName\\\":\\\"'+row['Position']+'\\\"'
        # Getting phone string
        if row['Phone'] == '':
            phonestr = ''
        else:
            phonestr = '{\\\"phoneNumber\\\":\\\"'+row['Phone']+'\\\"}'
        if row['org_uuid'] == '':
            orgstr = ''
        else:
            orgstr = '\\\"'+str(row['org_uuid'])+'\\\"'
        # Getting email string
        if row['Email'] == '':
            emstr = ''
        else:
            emstr = '\\\"'+str(row['Email'])+'\\\"'
            
        #Populating individual contact json template
        indtxt = ind_txt.replace(
            'UUID',str(row['uuid'])).replace(
            'NAME',row['FirstName']+' '+row['LastName']).replace(
            'ORGID',orgstr).replace(
            'PHONE',phonestr).replace(
            'EMAIL',emstr).replace(
            'POSITION',postr)

        # Opening template for contacts and populating, then putting that in json row for contact
        final_contacts.at[i,'json'] = contact_txt.replace('CURRENTDATETIME',currentdatetime
                                                         ).replace('MD_CONTACTID',contact_id
                                                                  ).replace('CONTACTJSON',indtxt)
        
# Look for duplicate contacts with mismatched info, find the longest json for each of those contacts
dupemail = final_contacts.groupby('Email').filter(lambda g: len(g) > 1).groupby(['Email', 'json']).head(1)
dupemail = dupemail.groupby('Email').filter(lambda g: len(g) > 1).groupby(['Email', 'json']).head(1)
dupemail = dupemail[['Email','json']]
dupemail['len'] = dupemail['json'].str.len()
dupemail = dupemail.sort_values('len', ascending=False).drop_duplicates('Email').sort_index().drop(columns='len').reset_index(drop=True)

# Replace json with longest json string per duplicate email
for i,row in dupemail.iterrows():
    email = row['Email']
    jsonstr = row['json']
    for i, row in final_contacts.iterrows():
        if str(row['Email'])==email:
            final_contacts.at[i,'json'] = jsonstr


# Skipping over contacts for products with pre-existing metadata that don't need records generated
if float(dmpvers)>=1.2:
    # If a user checks a box to indicate that they do not want new metadata produced for a product, metadata will not be created for that product
    nometa = df[(df.field.str.startswith('productSkipMeta'))&(df.value=='True')]
    nometalist = nometa['prod_num'].tolist()
elif float(dmpvers)<=1.1:
    # If a product is marked as existing and has the metadata URL field filled out, metadata will not be creating for that product
    existingproducts = df[(df.field.str.startswith('productExists'))&(df.value=='True')]
    existingproductslist = existingproducts['prod_num'].tolist()
    existingmeta = df[(df.field.str.startswith('productMetaURL'))&(df.value!='')]
    existingmetalist = existingproducts['prod_num'].tolist()
    nometalist = set(existingproductslist).intersection(existingmetalist)
    
final_contacts = final_contacts[~(final_contacts.product_num.isin(nometalist))]

display(final_contacts)

,Email,FirstName,LastName,Org,contact_type,NonDOI,Phone,Position,Role,product_num,json,uuid,org_uuid
0,devin_johnson@fws.gov,Devin,Johnson,,Author,,,,author,1,"{""id"": ""ea38mt9u"", ""attributes"": {""json"": ""{\""contactId\"":\""a077c537-0980-4d56-a19d-071e47074cc8\"",\""isOrganization\"":false,\""name\"":\""Devin Johnson\"",\""memberOfOrganization\"":[\""84557527-2349-436...",a077c537-0980-4d56-a19d-071e47074cc8,[84557527-2349-4365-98c2-2e1d16014c22]
2,devin_johnson@fws.gov,Devin,Johnson,,Creator,,,,originator,3,"{""id"": ""ea38mt9u"", ""attributes"": {""json"": ""{\""contactId\"":\""a077c537-0980-4d56-a19d-071e47074cc8\"",\""isOrganization\"":false,\""name\"":\""Devin Johnson\"",\""memberOfOrganization\"":[\""84557527-2349-436...",a077c537-0980-4d56-a19d-071e47074cc8,[84557527-2349-4365-98c2-2e1d16014c22]
3,devin_johnson@fws.gov,Devin,Johnson,,Custodian,False,,Wildlife Biologist,custodian,,"{""id"": ""ea38mt9u"", ""attributes"": {""json"": ""{\""contactId\"":\""a077c537-0980-4d56-a19d-071e47074cc8\"",\""isOrganization\"":false,\""name\"":\""Devin Johnson\"",\""memberOfOrganization\"":[\""84557527-2349-436...",a077c537-0980-4d56-a19d-071e47074cc8,[84557527-2349-4365-98c2-2e1d16014c22]
4,tamatha_patterson@fws.gov,Tamatha,Patterson,,DM,,,,custodian,,"{""id"": ""lk2rdmke"", ""attributes"": {""json"": ""{\""contactId\"":\""73ac8917-46cc-4d98-a7ec-5b5189e680b1\"",\""isOrganization\"":false,\""name\"":\""Tamatha A Patterson\"",\""memberOfOrganization\"":[\""f2d64d80-76...",73ac8917-46cc-4d98-a7ec-5b5189e680b1,"[f2d64d80-7641-4b87-b5a4-02250f27ad4a, 821858df-5d0e-445a-b027-014f5ef68782, 5a8ffdde-358e-4276-9289-596b7f23c22d]"
5,devin_johnson@fws.gov,Devin,Johnson,,Originator,False,,,originator,1,"{""id"": ""ea38mt9u"", ""attributes"": {""json"": ""{\""contactId\"":\""a077c537-0980-4d56-a19d-071e47074cc8\"",\""isOrganization\"":false,\""name\"":\""Devin Johnson\"",\""memberOfOrganization\"":[\""84557527-2349-436...",a077c537-0980-4d56-a19d-071e47074cc8,[84557527-2349-4365-98c2-2e1d16014c22]
9,jonah_withers@fws.gov,Jonah,Withers,,Other,,,,custodian,,"{""id"": ""mgk8t94s"", ""attributes"": {""json"": ""{\""contactId\"":\""960265ec-4fc7-47d6-a5b7-dcb346e2d8e5\"",\""isOrganization\"":false,\""name\"":\""Jonah Withers\"",\""memberOfOrganization\"":[\""23b9d60f-d181-440...",960265ec-4fc7-47d6-a5b7-dcb346e2d8e5,[23b9d60f-d181-4404-8262-18ae56bceab7]
10,devin_johnson@fws.gov,Devin,Johnson,,Steward,,,Wildlife Biologist,pointOfContact,,"{""id"": ""ea38mt9u"", ""attributes"": {""json"": ""{\""contactId\"":\""a077c537-0980-4d56-a19d-071e47074cc8\"",\""isOrganization\"":false,\""name\"":\""Devin Johnson\"",\""memberOfOrganization\"":[\""84557527-2349-436...",a077c537-0980-4d56-a19d-071e47074cc8,[84557527-2349-4365-98c2-2e1d16014c22]
11,daniel_bjornlie@fws.gov,Daniel,Bjornlie,,Trustee,,,,owner,,"{""id"": ""d42sape0"", ""attributes"": {""json"": ""{\""contactId\"":\""6e339443-d03c-4b2c-90fe-9359e25162ff\"",\""isOrganization\"":false,\""name\"":\""Dan Bjornlie\"",\""memberOfOrganization\"":[\""84557527-2349-4365...",6e339443-d03c-4b2c-90fe-9359e25162ff,[84557527-2349-4365-98c2-2e1d16014c22]
12,devin_johnson@fws.gov,Devin,Johnson,,Creator,,,,author,3,"{""id"": ""ea38mt9u"", ""attributes"": {""json"": ""{\""contactId\"":\""a077c537-0980-4d56-a19d-071e47074cc8\"",\""isOrganization\"":false,\""name\"":\""Devin Johnson\"",\""memberOfOrganization\"":[\""84557527-2349-436...",a077c537-0980-4d56-a19d-071e47074cc8,[84557527-2349-4365-98c2-2e1d16014c22]
13,,,,,,,,,administrator,,"{""id"": ""9i8qfij1"", ""attributes"": {""json"": ""{\""contactId\"":\""821858df-5d0e-445a-b027-014f5ef68782\"",\""isOrganization\"":true,\""name\"":\""U.S. Fish and Wildlife Service, Alaska Region\"",\""memberOfOrga...",821858df-5d0e-445a-b027-014f5ef68782,[]


In [10]:
# Iterate through org uuids in the lists and append to a new list, then remove duplicates and create a dataframe
orglist = []

for i, l in enumerate(final_contacts["org_uuid"]):
    if type(l)==list:
        for j in l:
            orglist.append(j)
    elif type(l)!=float:
        orglist.append(str(l))
        
orglist = [i for n, i in enumerate(orglist) if i not in orglist[:n]]
orgs = pd.DataFrame(orglist)
orgs.columns = ['org_uuid']

# Searching existing contacts for the organization info
for i, row in existing_contacts.iterrows():
    jsonstr = row['contact_json']
    for j, row in orgs.iterrows():
        corg = '{\\\"contactId\\\":\\\"'+str(row['org_uuid'])
        if corg!='nan' and corg in jsonstr:
            orgs.at[j,'org_json'] = jsonstr


# Preparing template for new contacts
with open(os.path.join(tmp_dir,'mdEd_contact.txt'), 'r') as handle:
    contact_txt = handle.read().strip()
    handle.close()
with open(os.path.join(tmp_dir,'org_contact.txt'), 'r') as handle:
    org_txt = handle.read().strip()
    handle.close()

# Filling out org_json for new organizations
for i, row in orgs.iterrows():
    if type(row['org_json'])==float:
        suuid = str(row['org_uuid'])
        for j, row in norg.iterrows():
            if str(row['uuid']) == suuid:
                # Generating an 8 digit ID for the record
                contact_id = str(uuid.uuid4()).lower()[:8]
                # Getting current datettime
                currentdatetime = str(datetime.now())
                currentdatetime = currentdatetime[:-3]
                currentdatetime = currentdatetime.replace(' ','T')+'Z'
                
                # Populating org contact template
                orgtxt = org_txt.replace('ORGUUID',str(row['uuid'])).replace('ORGNAME',row['Org'])
                # Populating mdEditor contact template
                orgs.at[i,'org_json'] = contact_txt.replace('CURRENTDATETIME',currentdatetime
                                                         ).replace('MD_CONTACTID',contact_id
                                                                  ).replace('CONTACTJSON',orgtxt)
                

# Getting org's organization uuid, by searching for uuids and removing the first in the list (contact's id)
# Then checking for new uuids and appending them to the existing dataframe
# This loops through and repeats a 3 times to make sure to get all nested organizations
newlist = orgs['org_json'].str.findall(
            '([a-fA-F0-9]{8}-[a-fA-F0-9]{4}-[a-fA-F0-9]{4}-[a-fA-F0-9]{4}-[a-fA-F0-9]{12})')[1:]
if len(newlist)>0:
    orglist2 = []
    for i, l in enumerate(newlist):
        if type(l)==list:
            for j in l:
                orglist2.append(j)
        elif type(l)!=float:
            orglist2.append(str(l))
    if len(orglist2)>0:
        orglist2 = [i for n, i in enumerate(orglist2) if i not in orglist2[:n]]
        for element in orgs['org_uuid']:
            if element in orglist2:
                orglist2.remove(element)
    if len(orglist2)>0:
        orgs2 = pd.DataFrame(orglist2)
        orgs2.columns = ['org_uuid']
        orgs = pd.concat([orgs,orgs2]).reset_index(drop=True)
        for i, row in existing_contacts.iterrows():
            jsonstr = row['contact_json']
            for j, row in orgs.iterrows():
                if type(row['org_json'])==float:
                    corg = '{\\\"contactId\\\":\\\"'+str(row['org_uuid'])
                    if corg!='nan' and corg in jsonstr:
                        orgs.at[j,'org_json'] = jsonstr
        newlist = orgs['org_json'].str.findall(
                    '([a-fA-F0-9]{8}-[a-fA-F0-9]{4}-[a-fA-F0-9]{4}-[a-fA-F0-9]{4}-[a-fA-F0-9]{12})')[1:]
        if len(newlist)>0:
            orglist2 = []
            for i, l in enumerate(newlist):
                if type(l)==list:
                    for j in l:
                        orglist2.append(j)
                elif type(l)!=float:
                    orglist2.append(str(l))
            if len(orglist2)>0:
                orglist2 = [i for n, i in enumerate(orglist2) if i not in orglist2[:n]]
                for element in orgs['org_uuid']:
                    if element in orglist2:
                        orglist2.remove(element)
            if len(orglist2)>0:
                orgs2 = pd.DataFrame(orglist2)
                orgs2.columns = ['org_uuid']
                orgs = pd.concat([orgs,orgs2]).reset_index(drop=True)
                for i, row in existing_contacts.iterrows():
                    jsonstr = row['contact_json']
                    for j, row in orgs.iterrows():
                        if type(row['org_json'])==float:
                            corg = '{\\\"contactId\\\":\\\"'+str(row['org_uuid'])
                            if corg!='nan' and corg in jsonstr:
                                orgs.at[j,'org_json'] = jsonstr
                newlist = orgs['org_json'].str.findall(
                            '([a-fA-F0-9]{8}-[a-fA-F0-9]{4}-[a-fA-F0-9]{4}-[a-fA-F0-9]{4}-[a-fA-F0-9]{12})')[1:]
                orgs = orgs.reset_index(drop=True)
                if len(newlist)>0:
                    orglist2 = []
                    for i, l in enumerate(newlist):
                        if type(l)==list:
                            for j in l:
                                orglist2.append(j)
                        elif type(l)!=float:
                            orglist2.append(str(l))
                    if len(orglist2)>0:
                        orglist2 = [i for n, i in enumerate(orglist2) if i not in orglist2[:n]]
                        for element in orgs['org_uuid']:
                            if element in orglist2:
                                orglist2.remove(element)
                    if len(orglist2)>0:
                        orgs2 = pd.DataFrame(orglist2)
                        orgs2.columns = ['org_uuid']
                        orgs = pd.concat([orgs,orgs2]).reset_index(drop=True)
                        for i, row in existing_contacts.iterrows():
                            jsonstr = row['contact_json']
                            for j, row in orgs.iterrows():
                                if type(row['org_json'])==float:
                                    corg = '{\\\"contactId\\\":\\\"'+str(row['org_uuid'])
                                    if corg!='nan' and corg in jsonstr:
                                        orgs.at[j,'org_json'] = jsonstr

orgs = orgs.reset_index(drop=True)
display(orgs)

,org_uuid,org_json
0,84557527-2349-4365-98c2-2e1d16014c22,"{""id"": ""9062earl"", ""attributes"": {""json"": ""{\""contactId\"":\""84557527-2349-4365-98c2-2e1d16014c22\"",\""isOrganization\"":true,\""name\"":\""Marine Mammals Management, Alaska Region\"",\""memberOfOrganizat..."
1,f2d64d80-7641-4b87-b5a4-02250f27ad4a,"{""id"": ""mj6gdc70"", ""attributes"": {""json"": ""{\""contactId\"":\""f2d64d80-7641-4b87-b5a4-02250f27ad4a\"",\""isOrganization\"":true,\""name\"":\""U.S. Fish and Wildlife Service\"",\""memberOfOrganization\"":[],\..."
2,821858df-5d0e-445a-b027-014f5ef68782,"{""id"": ""9i8qfij1"", ""attributes"": {""json"": ""{\""contactId\"":\""821858df-5d0e-445a-b027-014f5ef68782\"",\""isOrganization\"":true,\""name\"":\""U.S. Fish and Wildlife Service, Alaska Region\"",\""memberOfOrga..."
3,5a8ffdde-358e-4276-9289-596b7f23c22d,"{""id"": ""4hdou2nu"", ""attributes"": {""json"": ""{\""contactId\"":\""5a8ffdde-358e-4276-9289-596b7f23c22d\"",\""isOrganization\"":true,\""name\"":\""Migratory Bird Management Alaska\"",\""memberOfOrganization\"":[\..."
4,23b9d60f-d181-4404-8262-18ae56bceab7,"{""id"": ""olltg6o5"", ""attributes"": {""json"": ""{\""contactId\"":\""23b9d60f-d181-4404-8262-18ae56bceab7\"",\""isOrganization\"":true,\""name\"":\""Conservation Genetics Lab\"",\""memberOfOrganization\"":[\""d6ff24..."
5,d6ff2454-c6fa-4e13-a424-366d91efa60f,"{""id"": ""k6vbhv04"", ""attributes"": {""json"": ""{\""contactId\"":\""d6ff2454-c6fa-4e13-a424-366d91efa60f\"",\""isOrganization\"":true,\""name\"":\""Fisheries and Ecological Services, Alaska Region\"",\""memberOfO..."


In [11]:
# Combine orgnizations and individual contacts to create a list of jsons, for contacts in mdEditor file
# Region/Program org contacts removed as duplicates -- they already exist as
# permanent records in mdEditor (that's how region_program_contacts_path's
# lookup resolved them), so we don't want to re-create them here.
admin_uuids = admrows['uuid'].unique().tolist()
c = pd.concat([orgs[['org_json']].rename(columns={'org_json':'json'}),
              final_contacts[~(final_contacts.uuid.isin(admin_uuids))][['json']].drop_duplicates()]).reset_index(drop=True) #Removing duplication of Region/Program orgs
c = c.drop_duplicates().reset_index(drop=True) #Removing duplicates
display(c)

clist = c['json'].tolist()
    
contacts_json = ",".join(clist)


,json
0,"{""id"": ""9062earl"", ""attributes"": {""json"": ""{\""contactId\"":\""84557527-2349-4365-98c2-2e1d16014c22\"",\""isOrganization\"":true,\""name\"":\""Marine Mammals Management, Alaska Region\"",\""memberOfOrganizat..."
1,"{""id"": ""mj6gdc70"", ""attributes"": {""json"": ""{\""contactId\"":\""f2d64d80-7641-4b87-b5a4-02250f27ad4a\"",\""isOrganization\"":true,\""name\"":\""U.S. Fish and Wildlife Service\"",\""memberOfOrganization\"":[],\..."
2,"{""id"": ""9i8qfij1"", ""attributes"": {""json"": ""{\""contactId\"":\""821858df-5d0e-445a-b027-014f5ef68782\"",\""isOrganization\"":true,\""name\"":\""U.S. Fish and Wildlife Service, Alaska Region\"",\""memberOfOrga..."
3,"{""id"": ""4hdou2nu"", ""attributes"": {""json"": ""{\""contactId\"":\""5a8ffdde-358e-4276-9289-596b7f23c22d\"",\""isOrganization\"":true,\""name\"":\""Migratory Bird Management Alaska\"",\""memberOfOrganization\"":[\..."
4,"{""id"": ""olltg6o5"", ""attributes"": {""json"": ""{\""contactId\"":\""23b9d60f-d181-4404-8262-18ae56bceab7\"",\""isOrganization\"":true,\""name\"":\""Conservation Genetics Lab\"",\""memberOfOrganization\"":[\""d6ff24..."
5,"{""id"": ""k6vbhv04"", ""attributes"": {""json"": ""{\""contactId\"":\""d6ff2454-c6fa-4e13-a424-366d91efa60f\"",\""isOrganization\"":true,\""name\"":\""Fisheries and Ecological Services, Alaska Region\"",\""memberOfO..."
6,"{""id"": ""ea38mt9u"", ""attributes"": {""json"": ""{\""contactId\"":\""a077c537-0980-4d56-a19d-071e47074cc8\"",\""isOrganization\"":false,\""name\"":\""Devin Johnson\"",\""memberOfOrganization\"":[\""84557527-2349-436..."
7,"{""id"": ""lk2rdmke"", ""attributes"": {""json"": ""{\""contactId\"":\""73ac8917-46cc-4d98-a7ec-5b5189e680b1\"",\""isOrganization\"":false,\""name\"":\""Tamatha A Patterson\"",\""memberOfOrganization\"":[\""f2d64d80-76..."
8,"{""id"": ""mgk8t94s"", ""attributes"": {""json"": ""{\""contactId\"":\""960265ec-4fc7-47d6-a5b7-dcb346e2d8e5\"",\""isOrganization\"":false,\""name\"":\""Jonah Withers\"",\""memberOfOrganization\"":[\""23b9d60f-d181-440..."
9,"{""id"": ""d42sape0"", ""attributes"": {""json"": ""{\""contactId\"":\""6e339443-d03c-4b2c-90fe-9359e25162ff\"",\""isOrganization\"":false,\""name\"":\""Dan Bjornlie\"",\""memberOfOrganization\"":[\""84557527-2349-4365..."


## Distribution information
The next block of code prepares distribution and unique identifier information. If the RDR section of the DMP is completed, it will be listed in the distribution section of the metadata. If the project is listed as a Refuge project, the PRIMR ID will be extracted. If a ServCat reference URL is provided in the DMP, that will also get put into the distribution section of the metadata.

In [12]:
# Extracting RDR URL, project tracking code, and filepath from DMP template
if df[(df.field=='rdrURL')]['value'].values[0]!='':
    rdr = df[(df.field=='rdrURL')]['value'].values[0]
    rdr = rdr.replace('/','\\').replace('file:','')
    if rdr.endswith('\\'):
        rdr = rdr[:-1]

    if rdr.startswith('\\\\\\\\'):
        projectCode = rdr.split('\\\\')[-1]
        projectFolder = rdr.split('\\\\')[-2]
    elif rdr.startswith('\\\\'):
        projectCode = rdr.split('\\')[-1]
        projectFolder = rdr.split('\\')[-2]
    else:
        print('Warning, check RDR URL in DMP')

    display(projectCode)
    display(projectFolder)

    rdrfilestr = rdr.replace('\\','/')
    rdrfilestr = 'file:'+rdrfilestr
    display(rdrfilestr)
else: #if not using the RDR (e.g. Refuges), get program and extract PRIMR ID if applicable
    print('No RDR folder requested.')
    if df[(df.field=='FWSProgram')]['value'].values[0]=='National Wildlife Refuge System':
        projectFolder = 'nwrs'
        if df[(df.field=='projectUIDList')]['value'].values[0]:
            projectids = df[(df.field=='projectUIDList')]
            projectids['primr'] = projectids['value'].str.extract("(FF07[A-Za-z0-9]{6}[-][0-9]{3})", expand=False)
            primr = projectids['primr'].values[0]
            print('PRIMR ID: ',primr)
    elif df[(df.field=='FWSProgram')]['value'].values[0]=='Ecological Services' or 'Fish and Aquatic Conservation':
        projectFolder = 'fes'
        projectCode = 'NA'
    elif df[(df.field=='FWSProgram')]['value'].values[0]=='Migratory Birds':
        projectFolder = 'mbm'
        projectCode = 'NA'
    print(projectFolder)
    
    
# Getting ServCat info for project distribution section (assumes ServCat URL listed is project URL)
repo = df.copy()
repo['row'] = repo['field'].str.extract("([0-9]{1,})")
repo['field'] = repo['field'].str.replace("([0-9]{1,})", "", regex=True)
repo = repo[(repo.field.str.startswith('repo'))]
repo_t = repo.pivot(index='row', columns='field', values='value')
servcat = repo_t[(repo_t.repoName=='USFWS ServCat')&(repo_t.repoURL.str.contains('Reference'))].reset_index(drop=True)
display(servcat)

'fesmmm_028_Walrus_ Haulout_Satellite_Survey'

'fes'

'file://ifw7ro-file/datamgt/fes/fesmmm_028_Walrus_ Haulout_Satellite_Survey'

C:\Users\tpatterson\AppData\Local\Temp\1\ipykernel_27908\3997750169.py:46: FutureWarning: The behavior of Index.insert with object-dtype is deprecated, in a future version this will return an object-dtype Index instead of inferring a non-object dtype. To retain the old behavior, do `idx.insert(loc, item).infer_objects(copy=False)`
  repo_t = repo.pivot(index='row', columns='field', values='value')


field,repoName,repoURL


## Project/product dataframes
The following block of code transforms the information into dataframes. Each project and product row will become a metadata record within the initiated mdEditor file.

In [13]:
# Getting a unique list of product numbers
productcount = df[~(df.prod_num.isna())]['prod_num'].values.tolist()
productcount = list(set(productcount))
# productcount = productcount[1:]

# Getting a unique list of sample numbers
samplecount = df[~(df.sample_num.isna())]['sample_num'].values.tolist()
samplecount = list(set(samplecount))
# samplecount = samplecount[1:]

# Transposing the data frame so that the 'field' values are now column headers
df_t =  df.drop(columns=['prod_num','sample_num']).set_index('field').transpose().reset_index(drop=True)

# Extracting project columns into a separate data frame
project = df_t.filter(regex=('^(?!product)')).filter(regex=('^(?!sample)')).filter(regex=('^(?!contact)'))
project['uuid'] = uuid.uuid4()

# Getting unique project identifier code from RDR, to save as a variable for later
try:
    print('Project:',projectCode)
except:
    try:
        print('Project: ',primr)
    except:
        ('No RDR project code or PRIMR ID given.')

# Splitting keywords into a list (comma separated)
project['projectKeywords'] = project['projectKeywords'].str.split(',')
# Stripping out spaces that were between commas
project['projectKeywords'] = project['projectKeywords'].map(lambda x: list(map(str.strip, x)))

display(project)

# Combining product records into a single data frame, where each row represents a seperate product
products = pd.DataFrame()

for i in productcount:
    sub = df_t.filter(regex=('product')).filter(regex=('^(?!productMetaAuthor)')).filter(regex=('^(?!productOriginator)'))
    display(sub)
    rx = '([^0-9])'+str(i)+'$'
    sub = sub.filter(regex=rx).dropna(how='all')
    sub.columns = sub.columns.str.replace("([0-9]{1,})", "", regex=True)
    sub['productNo'] = i
    sub['uuid'] = uuid.uuid4()
    if sub['productTitle'][0]!='' and sub['productName'][0]!='': #Removing empty product tables based on title/name being present
        products = pd.concat([products,sub])
    
products['productNo'] = products['productNo'].astype(int)
products = products.sort_values('productNo').reset_index(drop=True)
products['productNo'] = products['productNo'].astype(str)

# Adding the DMP as a product
dmp_to_add = pd.Series({'productNo': str(len(products.index)+1),'productTitle': 'Data Management Plan',
             'productName': dmp.split('\\')[-1], 'productURL': dmp, 'productFileVersioning': '', 'productNew': 'True',
             'productExists': 'False', 'productMetaURL': '', 'productResourceType': 'document', 'productFormat': 'docx',
             'productFormatOther':'', 'productAbstract': 'Data management plan for '+project['projectTitle'][0]+'.',
             'productQualityAssurance': '', 'productQualityControl': '', 'productResources': '',
             'productMetaMaintenance': 'asNeeded', 'productRestriction': 'open access', 'productRestrictionJustification': '',
             'productAdditional': '', 'productSpatialURL': '', 'productSpatialDesc': '',
                        'productSpatialMatchesProject': 'True', 'uuid': uuid.uuid4()})
products = pd.concat([products,dmp_to_add.to_frame().T], ignore_index=True)

# Concatenating product QA/QC into a single string for lineage section
for i, row in products.iterrows():
    qa = row['productQualityAssurance'].replace('\\','/') #Replacing backslash for files with forward slash to avoid escape issues
    qc = row['productQualityControl'].replace('\\','/') #Replacing backslash for files with forward slash to avoid escape issues
    if qa!='NA' and qa!='N/A' and qa!='TBD' and len(qa)>0 and qa!='nan':
        qastr = 'Quality assurance: '+qa
        if qc!='NA' and qc!='N/A' and qc!='TBD' and len(qc)>0 and qc!='nan':
            qcstr = 'Quality control: '+qc
            products.at[i,'productLineage'] = qastr+'\\n\\n'+qcstr
        else:
            products.at[i,'productLineage'] = qastr
    else:
        if qc!='NA' and qc!='N/A' and qc!='TBD' and len(qc)>0 and qc!='nan':
            qcstr = 'Quality control: '+qc
            products.at[i,'productLineage'] = qcstr
        else:
            products.at[i,'productLineage'] = np.nan

print('There are',str(len(products.index)-1),'expected products and a data management plan.')
display(products)

# Combining sample records into a single data frame, where each row represents a seperate sample
samples = pd.DataFrame()

for i in samplecount:
    sub = df_t.filter(regex=('sample'))
    if not sub is None:
        rx = '([^0-9])'+str(i)+'$'
        sub = sub.filter(regex=rx)
#         sub = sub.replace(np.nan, '', inplace=True)
        sub.columns = sub.columns.str.replace("[0-9]+", "", regex=True)
        sub['sampleNo'] = i
        sub['uuid'] = uuid.uuid4()
        if sub['sampleName'][0]!='' and sub['sampleDescription'][0]!='': #Removing empty sample tables
            samples = pd.concat([samples,sub])
if len(samples.index)>0:
    samples = samples.sort_values('sampleNo').reset_index(drop=True)
    print('There are',max(samplecount),'expected physical samples.')
else:
    samples = samples.reset_index(drop=True)
    print('There are no expected physical samples.')
display(samples)

Project: fesmmm_028_Walrus_ Haulout_Satellite_Survey


field,dmpVersion,dmpCreateDate,dmpCreatorFirstName,dmpCreatorLastName,dmpCreatorEmail,dmpUpdateDate,dmpUpdaterFirstName,dmpUpdaterLastName,dmpUpdaterEmail,dmpSubmitDate,dmpDMFirstName,dmpDMLastName,dmpDMEmail,FWSProgram1,FWSProgram2,costCenter,projectTitle,projectStartDate,projectEndDate,projectOngoing,projectAbstract,projectKeywords,projectUIDList,projectSpatialDesc,projectSpatialURL,metaStandardMdJSON,metaStandardOther,metaStandardOtherType,dataStandard,storageOneDrive,storageOneDriveURL,storageTeams,storageTeamsURL,storageExternal,storageExternalLoc,storageNetwork,storageNetworkURL,storageOther,storageOtherLoc,backupFreq,backupFreqOther,filePermissions,dataReviewSchedule,rdrCheckbox,rdrURL,projectShortTitle,repoName,repoURL,recordsSchedule,recordsType,recordsDisposition,sourceDataDescription,sourceDataLoc,uuid
0,,2026-08-04,Devin,Johnson,Devin_johnson@fws.gov,2026-08-04,Devin,Johnson,Devin_johnson@fws.gov,,Tamatha,Patterson,Tamatha_patterson@fws.gov,Ecological Services,Ecological Services,FF07CAMM00,Pacific Walrus Haulout Occupancy Survey: 2016-2025 Sentinel-1 &amp; Sentinel-2 Satellite Imagery,2026-02-01,2026-08-04,True,"Pacific walrus (Odobenus rosmarus divergens) seasonally occupy coastal haulouts across their range, but seasonal space use patterns vary within and between years. Walruses are sensitive to human d...",[Walrus; haulout; occupancy; satellite; abundance],,"The Bering and Chukchi Seas and coasts, encompassing the Pacific walrus range within the United States",\\ifw7ro-file\datamgt\fes\fesmmm_028_Walrus_Haulout_Satellite_Survey\metadata\Walrus_BoundingBoxExtent.json,True,False,,Calendar Year,False,,False,,False,,False,,True,"\""\\ifw7ro-file.fws.doi.net\M:\WALRUS\DataBases\Haulout Occupancy”",Other,Completed Project,,Annual,True,\\ifw7ro-file\datamgt\fes\fesmmm_028_Walrus_ Haulout_Satellite_Survey,Walrus Mortality/Morbidity Database,USFWS ServCat,,,"These records document USFWS scientific research and investigation of wildlife, wildlife habitat, fish health, fishery biology, fishery management, and scientific research and investigation of con...",a. Study Case Files. Retention: TEMPORARY. Destroy 10 years after study is completed.; b. Historical Study Case Files. Completed studies case files selected annually by the project director as per...,Description:Spreadsheet containing haulout observations from satellite imagery,"\"" \""\\ifw7ro-file.fws.doi.net\ M:\WALRUS\DataBases\Haulout Occupancy”",8bee6788-b936-41ad-a183-d583b7da65bb


field,productNo1,productTitle1,productName1,productURL1,productFileVersioning1,productMetaURL1,productSkipMeta1,productResourceType1,productFormat1,productFormatOther1,productAbstract1,productQualityAssurance1,productQualityControl1,productResources1,productRestriction1,productRestrictionJustification1,productAdditional1,productSpatialMatchesProject1,productSpatialDesc1,productSpatialURL1,productNo2,productTitle2,productName2,productURL2,productFileVersioning2,productMetaURL2,productSkipMeta2,productResourceType2,productFormat2,productFormatOther2,productAbstract2,productQualityAssurance2,productQualityControl2,productResources2,productRestriction2,productRestrictionJustification2,productAdditional2,productSpatialMatchesProject2,productSpatialDesc2,productSpatialURL2
0,1,Pacific walrus haulout occupancy survey results - spreadsheet,HauloutSurveyResults_Sentinel_2016-2025,"\""\\ifw7ro-file\datamgt\fes\fesmmm_028_walrus_Mortality_Morbidity_Database\data\final_data\HauloutSurveyResults_Sentinel_2016-2025.csv\""",,"\""\\ifw7ro-file\datamgt\fes\fesmmm_028_walrus_Mortality_Morbidity_Database\data\final_data\HauloutSurveyResults_Sentinel_2016-2025_Metadata.pdf\""",False,collection,csv,,"Spreadsheet containing 1,785 discrete observations of walrus haulouts from various sources of satellite imagery, categorizing them into generalized size classes when possible.","This product compiles observations from trained wildlife biologists in the Marine Mammal Management program, and each observation was reviewed by a secondary observer following protocols establish...","This product compiles observations from trained wildlife biologists in the Marine Mammal Management program, and each observation was reviewed by a secondary observer following protocols establish...","NA, stored on shared drive and is &lt;1TB.",open access,,,True,"The Bering and Chukchi Seas and coasts, encompassing the Pacific walrus range within the United States",,2,Pacific Walrus Haulout Occupancy Survey - Report,PacificWalrusHauloutReport_Sentinel_2016-2025,"\""\\ifw7ro-file\datamgt\fes\fesmmm_028_walrus_Mortality_Morbidity_Database\documents\reports\PacificWalrusHauloutReport_Sentinel_2016-2025.pdf\""",,,True,report,pdf,,,"This product compiles observations from trained wildlife biologists in the Marine Mammal Management program, and each observation was reviewed by a secondary observer following protocols establish...","This product compiles observations from trained wildlife biologists in the Marine Mammal Management program, and each observation was reviewed by a secondary observer following protocols establish...","NA, stored on shared drive and is &lt;1TB.",open access,,,True,"The Bering and Chukchi Seas and coasts, encompassing the Pacific walrus range within the United States",


field,productNo1,productTitle1,productName1,productURL1,productFileVersioning1,productMetaURL1,productSkipMeta1,productResourceType1,productFormat1,productFormatOther1,productAbstract1,productQualityAssurance1,productQualityControl1,productResources1,productRestriction1,productRestrictionJustification1,productAdditional1,productSpatialMatchesProject1,productSpatialDesc1,productSpatialURL1,productNo2,productTitle2,productName2,productURL2,productFileVersioning2,productMetaURL2,productSkipMeta2,productResourceType2,productFormat2,productFormatOther2,productAbstract2,productQualityAssurance2,productQualityControl2,productResources2,productRestriction2,productRestrictionJustification2,productAdditional2,productSpatialMatchesProject2,productSpatialDesc2,productSpatialURL2
0,1,Pacific walrus haulout occupancy survey results - spreadsheet,HauloutSurveyResults_Sentinel_2016-2025,"\""\\ifw7ro-file\datamgt\fes\fesmmm_028_walrus_Mortality_Morbidity_Database\data\final_data\HauloutSurveyResults_Sentinel_2016-2025.csv\""",,"\""\\ifw7ro-file\datamgt\fes\fesmmm_028_walrus_Mortality_Morbidity_Database\data\final_data\HauloutSurveyResults_Sentinel_2016-2025_Metadata.pdf\""",False,collection,csv,,"Spreadsheet containing 1,785 discrete observations of walrus haulouts from various sources of satellite imagery, categorizing them into generalized size classes when possible.","This product compiles observations from trained wildlife biologists in the Marine Mammal Management program, and each observation was reviewed by a secondary observer following protocols establish...","This product compiles observations from trained wildlife biologists in the Marine Mammal Management program, and each observation was reviewed by a secondary observer following protocols establish...","NA, stored on shared drive and is &lt;1TB.",open access,,,True,"The Bering and Chukchi Seas and coasts, encompassing the Pacific walrus range within the United States",,2,Pacific Walrus Haulout Occupancy Survey - Report,PacificWalrusHauloutReport_Sentinel_2016-2025,"\""\\ifw7ro-file\datamgt\fes\fesmmm_028_walrus_Mortality_Morbidity_Database\documents\reports\PacificWalrusHauloutReport_Sentinel_2016-2025.pdf\""",,,True,report,pdf,,,"This product compiles observations from trained wildlife biologists in the Marine Mammal Management program, and each observation was reviewed by a secondary observer following protocols establish...","This product compiles observations from trained wildlife biologists in the Marine Mammal Management program, and each observation was reviewed by a secondary observer following protocols establish...","NA, stored on shared drive and is &lt;1TB.",open access,,,True,"The Bering and Chukchi Seas and coasts, encompassing the Pacific walrus range within the United States",


There are 2 expected products and a data management plan.


,productNo,productTitle,productName,productURL,productFileVersioning,productMetaURL,productSkipMeta,productResourceType,productFormat,productFormatOther,productAbstract,productQualityAssurance,productQualityControl,productResources,productRestriction,productRestrictionJustification,productAdditional,productSpatialMatchesProject,productSpatialDesc,productSpatialURL,uuid,productNew,productExists,productMetaMaintenance,productLineage
0,1,Pacific walrus haulout occupancy survey results - spreadsheet,HauloutSurveyResults_Sentinel_2016-2025,"\""\\ifw7ro-file\datamgt\fes\fesmmm_028_walrus_Mortality_Morbidity_Database\data\final_data\HauloutSurveyResults_Sentinel_2016-2025.csv\""",,"\""\\ifw7ro-file\datamgt\fes\fesmmm_028_walrus_Mortality_Morbidity_Database\data\final_data\HauloutSurveyResults_Sentinel_2016-2025_Metadata.pdf\""",False,collection,csv,,"Spreadsheet containing 1,785 discrete observations of walrus haulouts from various sources of satellite imagery, categorizing them into generalized size classes when possible.","This product compiles observations from trained wildlife biologists in the Marine Mammal Management program, and each observation was reviewed by a secondary observer following protocols establish...","This product compiles observations from trained wildlife biologists in the Marine Mammal Management program, and each observation was reviewed by a secondary observer following protocols establish...","NA, stored on shared drive and is &lt;1TB.",open access,,,True,"The Bering and Chukchi Seas and coasts, encompassing the Pacific walrus range within the United States",,c42c6371-c672-4d5e-baa7-985a1aa02b3b,NaN,NaN,NaN,"Quality assurance: This product compiles observations from trained wildlife biologists in the Marine Mammal Management program, and each observation was reviewed by a secondary observer following ..."
1,2,Pacific Walrus Haulout Occupancy Survey - Report,PacificWalrusHauloutReport_Sentinel_2016-2025,"\""\\ifw7ro-file\datamgt\fes\fesmmm_028_walrus_Mortality_Morbidity_Database\documents\reports\PacificWalrusHauloutReport_Sentinel_2016-2025.pdf\""",,,True,report,pdf,,,"This product compiles observations from trained wildlife biologists in the Marine Mammal Management program, and each observation was reviewed by a secondary observer following protocols establish...","This product compiles observations from trained wildlife biologists in the Marine Mammal Management program, and each observation was reviewed by a secondary observer following protocols establish...","NA, stored on shared drive and is &lt;1TB.",open access,,,True,"The Bering and Chukchi Seas and coasts, encompassing the Pacific walrus range within the United States",,6a546b3f-3bb8-4c9e-938a-94ba7f32147f,NaN,NaN,NaN,"Quality assurance: This product compiles observations from trained wildlife biologists in the Marine Mammal Management program, and each observation was reviewed by a secondary observer following ..."
2,3,Data Management Plan,AK_DMP_1.3_WalrusHallout_DJ.docm,C:\\Users\\tpatterson\\Downloads\\DevinJohnsonProject\\AK_DMP_1.3_WalrusHallout_DJ.docm,,,NaN,document,docx,,Data management plan for Pacific Walrus Haulout Occupancy Survey: 2016-2025 Sentinel-1 &amp; Sentinel-2 Satellite Imagery.,,,,open access,,,True,,,2d3d8d46-8340-46a9-83cf-db05976af563,True,False,asNeeded,NaN


There are no expected physical samples.


""


## Resulting mdEditor filename
The block of code below creates a filename for the initiated mdEditor file using the project code or PRIMR ID and today's date.

In [14]:
#Creating filename based on today's date and the RDR project code or PRIMR id

try:
    fn = projectCode+'-init-mdeditor-'+str(pd.to_datetime('today').date()).replace('-','')+'.json'
except:
    try:
        fn = primr+'-init-mdeditor-'+str(pd.to_datetime('today').date()).replace('-','')+'.json'
    except:
        fn = '-init-mdeditor-'+str(pd.to_datetime('today').date()).replace('-','')+'.json'

print(fn)

fesmmm_028_Walrus_ Haulout_Satellite_Survey-init-mdeditor-20260805.json


***
# Preparing mdEditor metadata
This section of the script creates mdEditor metadata from the dataframes above. If you are using this script only to push data management plans to the national SharePoint list, you can skip this section.

## Project metadata
The next several blocks of code prepare the project metadata record.

In [15]:
# Loading project-related JSON templates (txt files)
with open(os.path.join(tmp_dir,'project.txt'), 'r') as handle:
    proj_txt = handle.read().strip()
    handle.close()
with open(os.path.join(tmp_dir,'date_citation.txt'), 'r') as handle:
    date_cite_txt = handle.read().strip()
    handle.close()
with open(os.path.join(tmp_dir,'contactids.txt'), 'r') as handle:
    ids_txt = handle.read().strip()
    handle.close()
with open(os.path.join(tmp_dir,'contactroles.txt'), 'r') as handle:
    roles_txt = handle.read().strip()
    handle.close()
with open(os.path.join(tmp_dir,'extent.txt'), 'r') as handle:
    extent_txt = handle.read().strip()
    handle.close()
with open(os.path.join(tmp_dir,'associated_prod.txt'), 'r') as handle:
    assoc_txt = handle.read().strip()
    handle.close()
with open(os.path.join(tmp_dir,'primr_identifier.txt'), 'r') as handle:
    identifier_txt = handle.read().strip()
    handle.close()
with open(os.path.join(tmp_dir,'proj_dist.txt'), 'r') as handle:
    projdist_txt = handle.read().strip()
    handle.close()
with open(os.path.join(tmp_dir,'freetext_keywords.txt'), 'r') as handle:
    projkey_txt = handle.read().strip()
    handle.close()
    
#Setting project status based on start date, end date, and today's date
today = pd.to_datetime('today').date()
if project['projectOngoing'][0]=='False' and project['projectEndDate'][0]!='':
    if today>=datetime.strptime(project['projectEndDate'][0], '%Y-%m-%d').date():
        projectstatus = 'completed'
    elif today<datetime.strptime(project['projectEndDate'][0], '%Y-%m-%d').date():
        projectstatus = 'onGoing'
elif project['projectOngoing'][0]=='True' and project['projectEndDate'][0]=='':
    if today>=datetime.strptime(project['projectStartDate'][0], '%Y-%m-%d').date():
        projectstatus = 'onGoing'
    else:
        projectstatus = 'proposed'
elif project['projectOngoing'][0]=='True' and project['projectEndDate'][0]!='':
    if today>=datetime.strptime(project['projectStartDate'][0], '%Y-%m-%d').date() and today<datetime.strptime(
        project['projectEndDate'][0], '%Y-%m-%d').date():
        projectstatus = 'onGoing'
    elif today>=datetime.strptime(project['projectEndDate'][0], '%Y-%m-%d').date():
        projectstatus = 'completed'
    elif today<datetime.strptime(project['projectStartDate'][0], '%Y-%m-%d').date():
        projectstatus = 'proposed'

# Project contacts; all except for data manager, originators, DMP creator/updator
projcontacts = final_contacts[((final_contacts.product_num=='')|(final_contacts.Role=='author')) &
                              (final_contacts.contact_type!='DM') & (final_contacts.contact_type!='Creator') &
                              (final_contacts.contact_type!='Updater')
                             ][['Role','uuid']].groupby(['Role']).agg(lambda x: x.tolist()).reset_index()

# Putting point of contact at top of data frame
projcontacts['sort'] = range(1,len(projcontacts)+1)
projcontacts.loc[(projcontacts.Role=='pointOfContact'), 'sort'] = 0
projcontacts = projcontacts.sort_values("sort").drop('sort', axis=1)

# Creating JSON for metadata contact roles
metaauthors = projcontacts[(projcontacts.Role=='pointOfContact')|(projcontacts.Role=='author')|(projcontacts.Role=='publisher')]
metaauthjson = []
for i,row in metaauthors.iterrows():
    metaclist = []
    rstr = row['Role']
    for j, l in enumerate(metaauthors[(metaauthors.Role==rstr)]["uuid"]):
        for k in l:
            idstxt = ids_txt.replace('CONTACTUUID',str(k))
            metaclist += [idstxt]
    metacstr = ",".join(metaclist)
    metaauthstr = roles_txt.replace('CONTACTIDS',metacstr).replace('CONTACTROLE',row['Role'])
    metaauthjson += [metaauthstr]
metaauthjson =  ",".join(metaauthjson)

# Creating JSON strings for point of contacts (for citation)
# If the project is FES, principal investigator gets put into the citation, otherwise is left out
if projectFolder=='fes':
    pocs = projcontacts[(projcontacts.Role=='pointOfContact')|(projcontacts.Role=='publisher')|
                        (projcontacts.Role=='principalInvestigator')]
else:
    pocs = projcontacts[(projcontacts.Role=='pointOfContact')|(projcontacts.Role=='publisher')]

pocjson = []
for i,row in pocs.iterrows():
    pocclist = []
    rstr = row['Role']
    for j, l in enumerate(pocs[(pocs.Role==rstr)]["uuid"]):
        for k in l:
            idstxt = ids_txt.replace('CONTACTUUID',str(k))
            pocclist += [idstxt]
    poccstr = ",".join(pocclist)
    pocstr = roles_txt.replace('CONTACTIDS',poccstr).replace('CONTACTROLE',row['Role'])
    pocjson += [pocstr]
pocjson =  ",".join(pocjson)

# Creating JSON strings for project start and end dates
if project['projectEndDate'][0]!='':
    datecitejson = []
    datecitejson += [date_cite_txt.replace('DATEPLACECHOLDER',project['projectEndDate'][0]+'T00:00:00.000Z'
                                        ).replace('DATETYPEPLACEHOLDER','end')]
    datecitejson += [date_cite_txt.replace('DATEPLACECHOLDER',project['projectStartDate'][0]+'T00:00:00.000Z'
                                        ).replace('DATETYPEPLACEHOLDER','start')]
    datecitejson = ",".join(datecitejson)
    datejson = '\"endDateTime\": \"'+project['projectEndDate'][0]+'T00:00:00.000Z\",\"startDateTime\": \"'+project['projectStartDate'][0]+'T00:00:00.000Z\"'
else:
    datecitejson = date_cite_txt.replace('DATEPLACECHOLDER',project['projectStartDate'][0]+'T00:00:00.000Z'
                                        ).replace('DATETYPEPLACEHOLDER','start')
    datejson = '\"startDateTime\": \"'+project['projectStartDate'][0]+'T00:00:00.000Z\"'

# Creating JSON string for contact roles (not metadata authors)
roles = projcontacts[(projcontacts.Role!='author')&(projcontacts.Role!='distributor')&(projcontacts.Role!='publisher')]
rolejson = []
for i,row in roles.iterrows():
    roleclist = []
    rstr = row['Role']
    for j, l in enumerate(roles[(roles.Role==rstr)]["uuid"]): # Iterrating through list in uuid column
        for k in l:
            idstxt = ids_txt.replace('CONTACTUUID',str(k))
            roleclist += [idstxt]
    rolecstr = ",".join(roleclist)
    rolesstr = roles_txt.replace('CONTACTIDS',rolecstr).replace('CONTACTROLE',row['Role'])
    rolejson += [rolesstr]
rolejson =  ",".join(rolejson)

# Getting AK USFWS to assign as distribution contact for RDR folder
distuuid = str(final_contacts[(final_contacts.Role=='distributor')]['uuid'].values[0])

# Filling project metadata template
proj_json = proj_txt.replace('PROJECTUUID',str(project['uuid'][0])
                             ).replace('PROJECTABSTRACT',project['projectAbstract'][0]
                                      ).replace('PROJECTTITLE',project['projectTitle'][0]
                                               ).replace('PROJECTSHORTTITLE',project['projectShortTitle'][0]
                                                        ).replace('CURRENTDATE',str(date.today())+'T00:00:00.000Z'
                                                                 ).replace('METADATACONTACTS',metaauthjson
                                                                          ).replace('PROJECTDATESCITATION',datecitejson
                                                                                   ).replace('PROJECTCONTACTS',rolejson
                                                                                            ).replace('PROJECTDATES',datejson
                                                                                                      ).replace('DISTUUID',distuuid
                                                                                                               ).replace('POCCONTACTS',pocjson
                                                                                                                        ).replace('PROJECTSTATUS',projectstatus)

# Creating JSON string for spatial description, if it exists
# Creating JSON string for spatial extent, if it exists and is a geoJSON file
if project['projectSpatialDesc'][0]!='':
    projspatdesc = project['projectSpatialDesc'][0]
    if project['projectSpatialURL'][0]!='':
        geojstr = project['projectSpatialURL'][0].replace('/','\\')
        if geojstr.lower().endswith('json'):
            extenttxt = extent_txt.replace('EXTENTDESC','\"'+projspatdesc+'\"')
            with open(geojstr, 'r') as handle:
                geo_txt = handle.read().strip()
                handle.close()
            geo_txt = json.loads(geo_txt)
            geo_txt = json.dumps(geo_txt)
            extenttxt = extenttxt.replace('GEOJSON',geo_txt)
            proj_json = proj_json.replace('SPATIALEXTENT',extenttxt)
        else:
            extenttxt = '\"extent\":[{\"description\": \"'+projspatdesc+'\"}]'
            proj_json = proj_json.replace('SPATIALEXTENT',extenttxt)
    else:
        extenttxt = '\"extent\":[{\"description\": \"'+projspatdesc+'\"}]'
        proj_json = proj_json.replace('SPATIALEXTENT',extenttxt)
else:
    extenttxt = extent_txt.replace(',\"description\": EXTENTDESC','')
    if project['projectSpatialURL'][0]!='':
        geojstr = project['projectSpatialURL'][0].replace('/','\\')
        if os.path.exists(geojstr):
            if geojstr.lower().endswith('json'):
                with open(geojstr, 'r') as handle:
                    geo_txt = handle.read()
                    handle.close()
                geo_txt = json.loads(geo_txt)
                geo_txt = json.dumps(geo_txt)
                extenttxt = extenttxt.replace('GEOJSON',geo_txt)
                proj_json = proj_json.replace('SPATIALEXTENT',extenttxt)
            else:
                proj_json = proj_json.replace(',SPATIALEXTENT','')
        else:
            proj_json = proj_json.replace(',SPATIALEXTENT','')
    else:
        proj_json = proj_json.replace(',SPATIALEXTENT','')
    
        
# Loading JSON template for associated products for the project record, replacing placeholder words with data
# and iterating through the product dataframe for each record then adding it to the JSON list
assocprodjson = []
for i in products.index:
    if not products['productNo'][i] in nometalist: #Only adding association if metadata do not already exist
        assocprodstr = assoc_txt.replace('PRODUCTUID',str(products['uuid'][i]))
        if assocprodstr is not None:
            assocprodjson += [assocprodstr]
assocprodjson =  ",".join(assocprodjson)
assocprodjson = '\"associatedResource\": ['+assocprodjson+']'
if len(productcount)>0:
    proj_json = proj_json+','+assocprodjson
    
# Adding distribution for ServCat or RDR if ServCat not listed, or no distribution if neither are available
try:
    projdist_json = projdist_txt.replace('PROJDISTDESC','USFWS ServCat'
                                        ).replace('PROJECTURL',servcat['repoURL'].values[0]
                                                 ).replace('PROJDISTFUNCTION','information'
                                                          ).replace('DISTUUID','f2d64d80-7641-4b87-b5a4-02250f27ad4a')
    proj_json = proj_json.replace('PROJECTDISTRIBUTION',projdist_json)
except:
    try:
        projdist_json = projdist_txt.replace('PROJDISTDESC','Project folder on internal server, accessible to U.S. Fish and Wildlife Service staff.'
                                            ).replace('PROJECTURL',rdrfilestr
                                                     ).replace('PROJDISTFUNCTION','fileAccess'
                                                              ).replace('DISTUUID',str(final_contacts[(final_contacts.Role=='distributor')]['uuid'].values[0]))
        proj_json = proj_json.replace('PROJECTDISTRIBUTION',projdist_json)
    except:
        print('No RDR folder or ServCat project distribution.')

    
# Adding PRIMR unique survey identifier, if available
try:
    primrstr = identifier_txt.replace('PRIMRIDENTIFIER',primr)
    proj_json = proj_json.replace('PRIMRID',primrstr)
except:
    proj_json = proj_json.replace(',PRIMRID','')
    
    
# Adding keywords, if provided
# Creating JSON string for custom keywords
keywords = project['projectKeywords'][0]
keywordsjson = []
for i in keywords:
    keywordsjson.append('{\"keyword\": \"'+i+'\"}')
keywordsjson = ",".join(keywordsjson)
keywordsjson = projkey_txt.replace('PROJECTKEYWORDS',keywordsjson)
try:
    proj_json = proj_json.replace('PROJECTKEYWORDS',keywordsjson)
except:
    proj_json = proj_json.replace(',PROJECTKEYWORDS','')


# Closing off project JSON
proj_json = '\"schema\": {\"name\": \"mdJson\",\"version\": \"2.7.0\"},'+proj_json+'}'

In [16]:
if import_prof_schema == True:

    # Reading project profile information
    proj_prof_json = json.loads(urllib.request.urlopen(proj_profile).read())
    proj_schema_json = json.loads(urllib.request.urlopen(proj_schema).read())

    proj_prof_uuid1 = str(uuid.uuid4()).lower()[:8]
    proj_prof_uuid2 = str(uuid.uuid4()).lower()[:8]
    proj_schema_uuid = str(uuid.uuid4()).lower()[:8]

    # Getting current datettime
    currentdatetime = str(datetime.now())
    currentdatetime = currentdatetime[:-3]
    currentdatetime = currentdatetime.replace(' ','T')+'Z'

    with open(os.path.join(tmp_dir,'mdEd_profile.txt'), 'r') as handle:
        profile_txt = handle.read().strip()
        handle.close()

    projprofiletxt = profile_txt.replace(
        'PROFILE-JSON-ID',proj_prof_json['identifier']).replace(
        'PROFILEURL',proj_profile).replace(
        'PROFILEVERSION',proj_prof_json['version']).replace(
        'PROFILETITLE',proj_prof_json['title']).replace(
        'PROFILEID1',proj_prof_uuid1).replace(
        'PROFILEID2',proj_prof_uuid2).replace(
        'SCHEMAID',proj_schema_uuid).replace(
        'PROFILEJSON',str(json.dumps(proj_prof_json)).replace('\"','\\\"'))

    with open(os.path.join(tmp_dir,'mdEd_schema.txt'), 'r') as handle:
        schema_txt = handle.read().strip()
        handle.close()

    projschematxt = schema_txt.replace(
        'SCHEMAURL',proj_schema).replace(
        'SCHEMAJSON',str(json.dumps(proj_schema_json)).replace('\"','\\\"')).replace(
        'SCHEMAID',proj_schema_uuid).replace(
        'SCHEMAVERSION',proj_schema_json['version']).replace(
        'SCHEMADESCRIPTION',proj_schema_json['description']).replace(
        'SCHEMATITLE',proj_schema_json['title']).replace(
        'CURRENTDATETIME',currentdatetime)

    # display(json.loads(projprofiletxt))
    # display(json.loads(projschematxt))

In [17]:
if import_prof_schema == True:

    # Reading dictionary profile information
    dict_prof_json = json.loads(urllib.request.urlopen(dict_profile).read())
    dict_schema_json = json.loads(urllib.request.urlopen(dict_schema).read())

    dict_prof_uuid1 = str(uuid.uuid4()).lower()[:8]
    dict_prof_uuid2 = str(uuid.uuid4()).lower()[:8]
    dict_schema_uuid = str(uuid.uuid4()).lower()[:8]

    # Getting current datettime
    currentdatetime = str(datetime.now())
    currentdatetime = currentdatetime[:-3]
    currentdatetime = currentdatetime.replace(' ','T')+'Z'

    with open(os.path.join(tmp_dir,'mdEd_profile.txt'), 'r') as handle:
        profile_txt = handle.read().strip()
        handle.close()

    dictprofiletxt = profile_txt.replace(
        'PROFILE-JSON-ID',dict_prof_json['identifier']).replace(
        'PROFILEURL',dict_profile).replace(
        'PROFILEVERSION',dict_prof_json['version']).replace(
        'PROFILETITLE',dict_prof_json['title']).replace(
        'PROFILEID1',dict_prof_uuid1).replace(
        'PROFILEID2',dict_prof_uuid2).replace(
        'SCHEMAID',dict_schema_uuid).replace(
        'PROFILEJSON',str(json.dumps(dict_prof_json)).replace('\"','\\\"'))

    with open(os.path.join(tmp_dir,'mdEd_schema.txt'), 'r') as handle:
        schema_txt = handle.read().strip()
        handle.close()

    dictschematxt = schema_txt.replace(
        'SCHEMAURL',dict_schema).replace(
        'SCHEMAJSON',str(json.dumps(dict_schema_json)).replace('\"','\\\"')).replace(
        'SCHEMAID',dict_schema_uuid).replace(
        'SCHEMAVERSION',dict_schema_json['version']).replace(
        'SCHEMADESCRIPTION',dict_schema_json['description']).replace(
        'SCHEMATITLE',dict_schema_json['title']).replace(
        'CURRENTDATETIME',currentdatetime)

    # display(json.loads(projprofiletxt))
    # display(json.loads(projschematxt))

In [18]:
# Formatting mdJSON project for mdEditor file
finalprojjson = proj_json

# Escaping quotes and line breaks in project json string
finalprojjson = finalprojjson.replace('\"','\\\"')
finalprojjson = finalprojjson.replace('\\n','\\\\n')

with open(os.path.join(tmp_dir,'mdEd_record.txt'), 'r') as handle:
    record_txt = handle.read().strip()
    handle.close()
    
# Generating an 8 digit ID for the record
record_id = str(uuid.uuid4()).lower()[:8]

# Getting current datettime
currentdatetime = str(datetime.now())
currentdatetime = currentdatetime[:-3]
currentdatetime = currentdatetime.replace(' ','T')+'Z'

if import_prof_schema == True:
    recordtxt = record_txt.replace('CURRENTDATETIME',currentdatetime).replace('RECORDID',record_id
                                                                         ).replace('RECORDJSON','{'+finalprojjson+'}'
                                                                                  ).replace('PROFILEID',proj_prof_uuid1)
elif import_prof_schema == False and projectFolder == 'fes':
    recordtxt = record_txt.replace('CURRENTDATETIME',currentdatetime).replace('RECORDID',record_id
                                                                         ).replace('RECORDJSON','{'+finalprojjson+'}'
                                                                                  ).replace('PROFILEID','vtvuteah')
elif import_prof_schema == False and not projectFolder == 'fes':
    recordtxt = record_txt.replace('CURRENTDATETIME',currentdatetime).replace('RECORDID',record_id
                                                                         ).replace('RECORDJSON','{'+finalprojjson+'}'
                                                                                  ).replace('PROFILEID','org.adiwg.profile.full')
    
display(json.loads(recordtxt))

# display(recordtxt)

{'id': '3863e089',
 'attributes': {'profile': 'c2a8842e',
  'json': '{"schema": {"name": "mdJson","version": "2.7.0"},"metadata": {"metadataInfo": {"defaultMetadataLocale": {"characterSet": "UTF-8","country": "USA","language": "eng"},"metadataContact": [{"party":[{"contactId":"a077c537-0980-4d56-a19d-071e47074cc8"}],"role":"pointOfContact"},{"party":[{"contactId":"a077c537-0980-4d56-a19d-071e47074cc8"}],"role":"author"},{"party":[{"contactId":"821858df-5d0e-445a-b027-014f5ef68782"}],"role":"publisher"}],"metadataDate": [{"date": "2026-08-05T00:00:00.000Z","dateType": "creation"},{"date": "2026-08-05T00:00:00.000Z","dateType": "lastUpdate"}],"metadataIdentifier": {"identifier": "8bee6788-b936-41ad-a183-d583b7da65bb","namespace": "urn:uuid"},"metadataStatus": "initiated"},"resourceDistribution": [{"description": "Project folder on internal server, accessible to U.S. Fish and Wildlife Service staff.","distributor": [{"contact": {"party": [{"contactId": "821858df-5d0e-445a-b027-014f5ef6878

## Product metadata
The next several blocks of code loop through the product dataframe to create metadata records for each of the products.

In [19]:
# Notes: still need to figure out how to incorporate existing mdJSON files for selected products, if they exist

# Loading product-related JSON templates (txt files)
with open(os.path.join(tmp_dir,'product.txt'), 'r') as handle:
    prod_txt = handle.read().strip()
    handle.close()
with open(os.path.join(tmp_dir,'contactids.txt'), 'r') as handle:
    ids_txt = handle.read().strip()
    handle.close()
with open(os.path.join(tmp_dir,'contactroles.txt'), 'r') as handle:
    roles_txt = handle.read().strip()
    handle.close()
with open(os.path.join(tmp_dir,'extent.txt'), 'r') as handle:
    extent_txt = handle.read().strip()
    handle.close()
with open(os.path.join(tmp_dir,'open_access_constraints.txt'), 'r') as handle:
    openaccess_txt = handle.read().strip()
    handle.close()
with open(os.path.join(tmp_dir,'prod_dist.txt'), 'r') as handle:
    dist_txt = handle.read().strip()
    handle.close()
with open(os.path.join(tmp_dir,'dmp_keyword.txt'), 'r') as handle:
    dmp_key_txt = handle.read().strip()
    handle.close()
with open(os.path.join(tmp_dir,'freetext_keywords.txt'), 'r') as handle:
    prodkey_txt = handle.read().strip()
    handle.close()
with open(os.path.join(tmp_dir,'lineage.txt'), 'r') as handle:
    lin_txt = handle.read().strip()
    handle.close()
    
# Replacing data manager UUID placeholder in distribution
dist_txt = dist_txt.replace('DMUUID',distuuid)
    
# Getting project metadata info (start date) to put in associated section
metadatecitejson = date_cite_txt.replace('DATEPLACECHOLDER',project['dmpSubmitDate'][0]+'T00:00:00.000Z'
                                        ).replace('DATETYPEPLACEHOLDER','start')

# Inserting associated project info into the product JSON template
prod_txt = prod_txt.replace('METADATACONTACTS',metaauthjson).replace('PROJECTDATESCITATION',metadatecitejson).replace(
    'PROJECTDATES',datecitejson).replace('PROJECTTITLE',project['projectTitle'][0]).replace(
    'PROJECTSHORTTITLE',project['projectShortTitle'][0]).replace('PROJECTUUID',str(project['uuid'][0])).replace(
    'POCCONTACTS',pocjson)

# Getting AK USFWS to assign as distribution contact for RDR folder
distuuid = str(final_contacts[(final_contacts.Role=='distributor')]['uuid'].values[0])

# Creating an empty list to store individual product JSON strings  
prod_json = []

# Looping through products to create JSON
for i in products.index:
    if not products['productNo'][i] in nometalist: #Skipping over product if metadata does not need to be generated
        print('--')
        print('Product title: '+products['productTitle'][i])
        prodnum = str(i+1) # Product number (index starts at 0)
        prodtxt = prod_txt # copying over base JSON string from above

        prodcontacts = final_contacts[(final_contacts.product_num==prodnum) |
                                      (final_contacts.product_num=='')
                                     ][['contact_type','Role','uuid']].groupby(['Role']).agg(lambda x: x.tolist()).reset_index()
        # Putting point of contact at top of data frame
        prodcontacts['sort'] = range(1,len(prodcontacts)+1)
        prodcontacts.loc[(prodcontacts.Role=='pointOfContact'), 'sort'] = 0
        prodcontacts = prodcontacts.sort_values("sort").drop('sort', axis=1)

        # citation contacts get transferred directly from the project citation json in previous script block

        # Looping through metadata authors to add to metadata section
        prodmetaauthors = prodcontacts[(prodcontacts.Role=='author')|(prodcontacts.Role=='pointOfContact')|(prodcontacts.Role=='publisher')]
        prodmetaauthjson = []
        for j,row in prodmetaauthors.iterrows():
            metaclist = []
            rstr = row['Role']
            for k, l in enumerate(prodmetaauthors[(prodmetaauthors.Role==rstr)]["uuid"]):
                for m in l:
                    prodidstxt = ids_txt.replace('CONTACTUUID',str(m))
                    metaclist += [prodidstxt]
            prodmetacstr = ",".join(metaclist)
            prodmetaauthstr = roles_txt.replace('CONTACTIDS',prodmetacstr).replace('CONTACTROLE',row['Role'])
            prodmetaauthjson += [prodmetaauthstr]
        prodmetaauthjson =  ",".join(prodmetaauthjson)

        # Looping through all roles to add to contacts for the product, except metadata authors, publisher, and distributor
        prodroles = prodcontacts[(prodcontacts.Role!='author')&(prodcontacts.Role!='publisher')&(prodcontacts.Role!='distributor')]
        prodrolejson = []
        for n,row in prodroles.iterrows():
            roleclist = []
            rstr = row['Role']
            for o, l in enumerate(prodroles[(prodroles.Role==rstr)]["uuid"]): # Iterrating through list in uuid column
                for p in l:
                    idstxt = ids_txt.replace('CONTACTUUID',str(p))
                    roleclist += [idstxt]
            rolecstr = ",".join(roleclist)
            prodrolesstr = roles_txt.replace('CONTACTIDS',rolecstr).replace('CONTACTROLE',row['Role'])
            prodrolejson += [prodrolesstr]
        prodrolejson =  ",".join(prodrolejson)
        
        # Adding dmp keyword to DMP product and marking status as completed
        print(products['productTitle'][i])
        if products['productTitle'][i]=='Data Management Plan':
            # Using already created keywords JSON string from project section to add to GCMD keyword
            try:
                prodkeywordsjson = keywordsjson[11:-1] # Removing extra keyword list to embed within other keyword list
                dmpkeytxt = dmp_key_txt.replace('PROJECTKEYWORDS',prodkeywordsjson)
            except:
                dmpkeytxt = dmp_key_txt.replace(',PROJECTKEYWORDS','')
            prodtxt = prodtxt.replace('DMPKEYWORD',dmpkeytxt)
            prodtxt = prodtxt.replace('\"status\": [\"proposed\"]','\"status\": [\"completed\"]')
        # Or adding project keywords to the products, if provided
        else:
            # Using already created keywords JSON string from project section
            try:
                prodtxt = prodtxt.replace('DMPKEYWORD',keywordsjson)
            except:
                prodtxt = prodtxt.replace(',DMPKEYWORD','')
                
        # Adding lineage statement for QA/QC
        if incl_quality == True and products['productLineage'][i]!='nan' and pd.isnull(products['productLineage'][i])==False:
            print('QA/QC statement included in lineage section')
            # Build json string for lineage statement
            lintxt = lin_txt.replace('LINEAGESTATEMENT',products['productLineage'][i])
            try:
                prodtxt = prodtxt.replace('PRODUCTLINEAGE',lintxt)
            except:
                prodtxt = prodtxt.replace(',PRODUCTLINEAGE','')
        else:
            prodtxt = prodtxt.replace(',PRODUCTLINEAGE','')

        # Adding DMP create/update dates to citation section
        if products['productTitle'][i]=='Data Management Plan':
            proddatejson = '\"timePeriod\": {\"startDateTime\": \"'+project['dmpCreateDate'][0]+'T00:00:00.000Z\"}'
            print(proddatejson)
            if project['dmpUpdateDate'][0]!='':
                proddatecitejson = []
                proddatecitejson += [date_cite_txt.replace('DATEPLACECHOLDER',project['dmpUpdateDate'][0]+'T00:00:00.000Z'
                                                    ).replace('DATETYPEPLACEHOLDER','lastUpdate')]
                proddatecitejson += [date_cite_txt.replace('DATEPLACECHOLDER',project['dmpCreateDate'][0]+'T00:00:00.000Z'
                                                    ).replace('DATETYPEPLACEHOLDER','creation')]
                proddatecitejson = ",".join(proddatecitejson)
            else:
                proddatecitejson = date_cite_txt.replace('DATEPLACECHOLDER',project['dmpCreateDate'][0]+'T00:00:00.000Z'
                                                    ).replace('DATETYPEPLACEHOLDER','creation')
            prodtxt = prodtxt.replace('PRODUCTDATESCITATION',proddatecitejson).replace('PRODUCTDATE',proddatejson)
        else:
            prodtxt = prodtxt.replace('\"date\": [PRODUCTDATESCITATION],','').replace(',PRODUCTDATE','')


        # Replacing holding text values in json product template
        prodtxt = prodtxt.replace('PRODUCTUUID',str(products['uuid'][i])
                         ).replace('PRODUCTRESOURCETYPE',products['productResourceType'][i]
                                  ).replace('PRODUCTNAME',products['productName'][i]
                                           ).replace('PRODUCTABSTRACT',products['productAbstract'][i]
                                                    ).replace('PRODUCTCONTACTS',prodrolejson
                                                             ).replace('PRODUCTTITLE',products['productTitle'][i]
                                                                      ).replace('CURRENTDATE',str(date.today())+'T00:00:00.000Z'
                                                                               ).replace('PRODMETADATAAUTH',prodmetaauthjson
                                                                                        ).replace('POCCONTACTS',pocjson)

        # Checking for spatial extent feature and adding it, if geoJSON exists
        prodspat = products['productSpatialURL'][i].replace('/','\\')
        prodspatdesc = products['productSpatialDesc'][i]
        # If the product spatial matches the project spatial...
        if products['productSpatialMatchesProject'][i]=='True':
            if project['projectSpatialDesc'][0]!='': #... and there's a description
                projspatdesc = project['projectSpatialDesc'][0]
                if project['projectSpatialURL'][0]!='':
                    prodgeojstr = project['projectSpatialURL'][0].replace('/','\\')
                    if os.path.exists(prodgeojstr): # and the file is accessible
                        if prodgeojstr.lower().endswith('json'): #... and the url is a json
                            prodextenttxt = extent_txt.replace('EXTENTDESC','\"'+projspatdesc+'\"')
                            with open(prodgeojstr, 'r') as handle:
                                prodgeo_txt = handle.read().strip()
                                handle.close()
                            prodgeo_txt = json.loads(prodgeo_txt)
                            prodgeo_txt = json.dumps(prodgeo_txt)
                            prodextenttxt = prodextenttxt.replace('GEOJSON',prodgeo_txt)
                            prodtxt = prodtxt.replace('EXTENT',prodextenttxt)
                        else: #... and the url is not a json
                            prodextenttxt = '\"extent\":[{\"description\": \"'+projspatdesc+'\"}]'
                            prodtxt = prodtxt.replace('EXTENT',prodextenttxt)
                    else:
                        prodextenttxt = '\"extent\":[{\"description\": \"'+projspatdesc+'\"}]'
                        prodtxt = prodtxt.replace('EXTENT',prodextenttxt)
                else: #... and there is no spatial url
                    prodextenttxt = '\"extent\":[{\"description\": \"'+projspatdesc+'\"}]'
                    prodtxt = prodtxt.replace('EXTENT',prodextenttxt)
            else: #... and there's no description
                prodextenttxt = extent_txt.replace(',\"description\": EXTENTDESC','')
                if project['projectSpatialURL'][0]!='':
                    prodgeojstr = project['projectSpatialURL'][0].replace('/','\\')
                    if os.path.exists(prodgeojstr): # and the file is accessible
                        if prodgeojstr.lower().endswith('json'): #... and the url is a json
                            with open(prodgeojstr, 'r') as handle:
                                prodgeo_txt = handle.read().strip()
                                handle.close()
                            prodgeo_txt = json.loads(prodgeo_txt)
                            prodgeo_txt = json.dumps(prodgeo_txt)
                            prodextenttxt = prodextenttxt.replace('GEOJSON',prodgeo_txt)
                            prodtxt = prodtxt.replace('EXTENT',prodextenttxt)
                        else:
                            prodtxt = prodtxt.replace(',EXTENT','')
                    else: #... and the url is not a json
                        prodtxt = prodtxt.replace(',EXTENT','')
                else: #... and there is no url
                    prodtxt = prodtxt.replace(',EXTENT','')
        else: # the product spatial does not match the project spatial...
            if prodspatdesc!='': #... and there is a description
                prodextenttxt = extent_txt.replace('EXTENTDESC','\"'+prodspatdesc+'\"')
                if prodspat!='': #... and there is a spatial url
                    if os.path.exists(prodspat): # and the file is accessible
                        if prodspat.lower().endswith('json'): #... and the url is a json
                            with open(prodspat, 'r') as handle:
                                prodgeo_txt = handle.read().strip()
                                handle.close()
                            prodgeo_txt = json.loads(prodgeo_txt)
                            prodgeo_txt = json.dumps(prodgeo_txt)
                            prodextenttxt = prodextenttxt.replace('GEOJSON',prodgeo_txt)
                            prodtxt = prodtxt.replace('EXTENT',prodextenttxt)
                        else: # not a geo json or is empty
                            print('Warning: supplied spatial data file is not a JSON; no extent added.')
                            print('     '+prodspat)
                            prodextenttxt = '\"extent\":[{\"description\": \"'+prodspatdesc+'\"}]'
                            prodtxt = prodtxt.replace('EXTENT',prodextenttxt)
                    else: # but the file is not accessible
                        print('Warning: supplied spatial data file does not exist or is not accessible. No extent added.')
                        print('     '+prodspat)
                        prodextenttxt = '\"extent\":[{\"description\": \"'+prodspatdesc+'\"}]'
                        prodtxt = prodtxt.replace('EXTENT',prodextenttxt)
                else: #... and there is no spatial url
                    prodextenttxt = '\"extent\":[{\"description\": \"'+prodspatdesc+'\"}]'
                    prodtxt = prodtxt.replace('EXTENT',prodextenttxt)
            else: #... and there is no description
                prodextenttxt = extent_txt.replace(',\"description\": EXTENTDESC','')
                if prodspat!='': #... and there is a spatial url
                    if os.path.exists(prodspat): # and the file is accessible
                        if prodspat.lower().endswith('json'): #... and the url is a json
                            with open(prodspat, 'r') as handle:
                                prodgeo_txt = handle.read().strip()
                                handle.close()
                            prodgeo_txt = json.loads(prodgeo_txt)
                            prodgeo_txt = json.dumps(prodgeo_txt)
                            prodextenttxt = prodextenttxt.replace('GEOJSON',prodgeo_txt)
                            prodtxt = prodtxt.replace('EXTENT',prodextenttxt)
                        else: # not a geo json or is empty
                            print('Warning: supplied spatial data file is not a JSON; no extent added.')
                            print('     '+prodspat)
                            prodtxt = prodtxt.replace(',EXTENT','')
                    else: # but the file is not accessible
                        print('Warning: supplied spatial data file does not exist or is not accessible. No extent added.')
                        print('     '+prodspat)
                        prodtxt = prodtxt.replace(',EXTENT','')
                else: #... and there is no spatial url
                    prodtxt = prodtxt.replace(',EXTENT','')


        # Adding default constraint text if product is marked as open access
        prodcons = products['productRestriction'][i]
        if prodcons == 'open access' and incl_disclaimer == True:
            prodtxt = prodtxt.replace('OPENACCESS',openaccess_txt)
        else:
            prodtxt = prodtxt.replace(',OPENACCESS','')

        # Adding distribution if there is an RDR archived file for the product
        if products['productTitle'][i]=='Data Management Plan':
            produrl = dmp
            if 'ifw7ro-file.fws.doi.net/datamgt' in produrl or 'ifw7ro-file.fws.doi.net\\datamgt' in produrl:
                #If DMP is in RDR, metadata status = completed; if not, metadata status=initiated because no distribution
                prodtxt = prodtxt.replace('\"metadataStatus\": \"initiated\"','\"metadataStatus\": \"completed\"')
        else:
            produrl = products['productURL'][i]
        produrl = produrl.replace('\\','/')
        if produrl!='':
            if 'ifw7ro-file.fws.doi.net/datamgt' in produrl or 'ifw7ro-file.fws.doi.net\\datamgt' in produrl:
                if not produrl.startswith('//') and not produrl.startswith('file://'):
                    produrlstr = 'file://'+produrl
                elif produrl.startswith('//'):
                    produrlstr = 'file:'+produrl
                elif produrl.startswith('file://'):
                    produrlstr = produrl
                else:
                    produrlstr = produrl
                    print('Warning: check URL for product '+products['productName'][i])
                    print('     '+produrlstr)
                if os.path.exists(produrl.replace('/','\\')): # Checking to see if the file exists
                    disttxt = dist_txt.replace('PRODUCTURL',produrlstr).replace('DISTUUID',distuuid)
                    prodtxt = prodtxt.replace('PRODUCTDISTRIBUTION',disttxt)
                    print('RDR distribution added.')
                    print('     '+produrlstr)
                else:
                    print('Warning: check URL for product '+products['productName'][i])
                    print('     '+produrlstr)
                    print('')
                    prodtxt = prodtxt.replace(',PRODUCTDISTRIBUTION','')
            else:
                prodtxt = prodtxt.replace(',PRODUCTDISTRIBUTION','')
                print('Product file is not in RDR; no distribution added.')
                print('     '+produrl)
                print('')
        else:
            prodtxt = prodtxt.replace(',PRODUCTDISTRIBUTION','')
            print('No product file provided; no distributon added.')

        # Closing off the product JSON string
        prodtxt = '{\"schema\": {\"name\": \"mdJson\",\"version\": \"2.7.0\"},'+prodtxt+'}}'
        prod_json += [prodtxt] #Adding json string for each product into the json string list for products

    else:
        print('--')
        print(products['productTitle'][i],': Product metadata already exists. mdEditor product record not generated.')

--
Product title: Pacific walrus haulout occupancy survey results - spreadsheet
Pacific walrus haulout occupancy survey results - spreadsheet
QA/QC statement included in lineage section
Product file is not in RDR; no distribution added.
     /"//ifw7ro-file/datamgt/fes/fesmmm_028_walrus_Mortality_Morbidity_Database/data/final_data/HauloutSurveyResults_Sentinel_2016-2025.csv/"

--
Pacific Walrus Haulout Occupancy Survey - Report : Product metadata already exists. mdEditor product record not generated.
--
Product title: Data Management Plan
Data Management Plan
"timePeriod": {"startDateTime": "2026-08-04T00:00:00.000Z"}
Product file is not in RDR; no distribution added.
     C://Users//tpatterson//Downloads//DevinJohnsonProject//AK_DMP_1.3_WalrusHallout_DJ.docm



In [20]:
if import_prof_schema == True:

    # Reading project profile information
    prod_prof_json = json.loads(urllib.request.urlopen(prod_profile).read())
    prod_schema_json = json.loads(urllib.request.urlopen(prod_schema).read())

    prod_prof_uuid1 = str(uuid.uuid4()).lower()[:8]
    prod_prof_uuid2 = str(uuid.uuid4()).lower()[:8]
    prod_schema_uuid = str(uuid.uuid4()).lower()[:8]

    # Getting current datettime
    currentdatetime = str(datetime.now())
    currentdatetime = currentdatetime[:-3]
    currentdatetime = currentdatetime.replace(' ','T')+'Z'

    with open(os.path.join(tmp_dir,'mdEd_profile.txt'), 'r') as handle:
        profile_txt = handle.read().strip()
        handle.close()

    prodprofiletxt = profile_txt.replace(
        'PROFILE-JSON-ID',prod_prof_json['identifier']).replace(
        'PROFILEURL',prod_profile).replace(
        'PROFILEVERSION',prod_prof_json['version']).replace(
        'PROFILETITLE',prod_prof_json['title']).replace(
        'PROFILEID1',prod_prof_uuid1).replace(
        'PROFILEID2',prod_prof_uuid2).replace(
        'SCHEMAID',prod_schema_uuid).replace(
        'PROFILEJSON',str(json.dumps(prod_prof_json)).replace('\"','\\\"'))

    with open(os.path.join(tmp_dir,'mdEd_schema.txt'), 'r') as handle:
        schema_txt = handle.read().strip()
        handle.close()

    prodschematxt = schema_txt.replace(
        'SCHEMAURL',prod_schema).replace(
        'SCHEMAJSON',str(json.dumps(prod_schema_json)).replace('\"','\\\"')).replace(
        'SCHEMAID',prod_schema_uuid).replace(
        'SCHEMAVERSION',prod_schema_json['version']).replace(
        'SCHEMADESCRIPTION',prod_schema_json['description']).replace(
        'SCHEMATITLE',prod_schema_json['title']).replace(
        'CURRENTDATETIME',currentdatetime)

    # display(json.loads(prodprofiletxt))
    # display(json.loads(prodschematxt))

In [21]:
# Formatting mdJSON products for mdEditor file
finalprodjson = prod_json

# Loading template for mdEditor records
with open(os.path.join(tmp_dir,'mdEd_record.txt'), 'r') as handle:
    record_txt = handle.read().strip()
    handle.close()
    
prodjson_list = []

for i in finalprodjson:
    # Escaping quotes in product json string
    prodjsontxt = str(i).replace('\\','\\\\').replace('\"','\\\"').replace('\\n','\\\\n').replace('\\\\\\n','\\\\n') #Some text fields have quotes, must undo replacement of already escaped txt
    # Generating an 8 digit ID for the record
    record_id = str(uuid.uuid4()).lower()[:8]
    # Getting current datettime
    currentdatetime = str(datetime.now())
    currentdatetime = currentdatetime[:-3]
    currentdatetime = currentdatetime.replace(' ','T')+'Z'
    if import_prof_schema == True:
        prodtxt = record_txt.replace(
            'CURRENTDATETIME',currentdatetime).replace(
            'RECORDID',record_id).replace(
            'RECORDJSON',prodjsontxt).replace(
            'PROFILEID',prod_prof_uuid1)
    elif import_prof_schema == False and projectFolder == 'fes':
        prodtxt = record_txt.replace(
            'CURRENTDATETIME',currentdatetime).replace(
            'RECORDID',record_id).replace(
            'RECORDJSON',prodjsontxt).replace(
            'PROFILEID','c41g60sl')
    elif import_prof_schema == False and not projectFolder=='fes':
        prodtxt = record_txt.replace(
            'CURRENTDATETIME',currentdatetime).replace(
            'RECORDID',record_id).replace(
            'RECORDJSON',prodjsontxt).replace(
            'PROFILEID','org.adiwg.profile.full')
    prodjson_list.append(prodtxt)
    display(json.loads(prodtxt))
    
prodrecordtxt = ','.join(prodjson_list)

{'id': 'd29e2aa7',
 'attributes': {'profile': 'b224a06b',
  'json': '{"schema": {"name": "mdJson","version": "2.7.0"},"metadata":{"metadataInfo":{"metadataIdentifier":{"identifier":"c42c6371-c672-4d5e-baa7-985a1aa02b3b","namespace":"urn:uuid"},"metadataStatus": "initiated","defaultMetadataLocale":{"language":"eng","characterSet":"UTF-8","country":"USA"},"metadataContact": [{"party":[{"contactId":"a077c537-0980-4d56-a19d-071e47074cc8"}],"role":"pointOfContact"},{"party":[{"contactId":"a077c537-0980-4d56-a19d-071e47074cc8"}],"role":"author"},{"party":[{"contactId":"821858df-5d0e-445a-b027-014f5ef68782"}],"role":"publisher"}],"metadataDate": [{"date": "2026-08-05T00:00:00.000Z","dateType": "creation"},{"date": "2026-08-05T00:00:00.000Z","dateType": "lastUpdate"}]},"resourceInfo":{"resourceType":[{"type":"collection","name":"HauloutSurveyResults_Sentinel_2016-2025"}],"status": ["proposed"],"citation":{"title":"Pacific walrus haulout occupancy survey results - spreadsheet","responsibleParty

{'id': 'efe691a3',
 'attributes': {'profile': 'b224a06b',
  'json': '{"schema": {"name": "mdJson","version": "2.7.0"},"metadata":{"metadataInfo":{"metadataIdentifier":{"identifier":"2d3d8d46-8340-46a9-83cf-db05976af563","namespace":"urn:uuid"},"metadataStatus": "initiated","defaultMetadataLocale":{"language":"eng","characterSet":"UTF-8","country":"USA"},"metadataContact": [{"party":[{"contactId":"a077c537-0980-4d56-a19d-071e47074cc8"}],"role":"pointOfContact"},{"party":[{"contactId":"a077c537-0980-4d56-a19d-071e47074cc8"}],"role":"author"},{"party":[{"contactId":"821858df-5d0e-445a-b027-014f5ef68782"}],"role":"publisher"}],"metadataDate": [{"date": "2026-08-05T00:00:00.000Z","dateType": "creation"},{"date": "2026-08-05T00:00:00.000Z","dateType": "lastUpdate"}]},"resourceInfo":{"resourceType":[{"type":"document","name":"AK_DMP_1.3_WalrusHallout_DJ.docm"}],"status": ["completed"],"citation":{"date": [{"date": "2026-08-04T00:00:00.000Z","dateType": "lastUpdate"},{"date": "2026-08-04T00:00

## Initiated mdEditor File
The next block of code combines the project record, product records, and (optionally) profiles and schemas to write an mdEditor file.

In [22]:
if import_prof_schema == True:
    print('Schema and profiles included in mdEditor file.')
    finaljson = '{\"data\": ['+recordtxt+','+prodrecordtxt+','+contacts_json+','+projprofiletxt+','+projschematxt+','+prodprofiletxt+','+prodschematxt+','+dictprofiletxt+','+dictschematxt+']}'
#     finaljson = '{\"data\": ['+prodrecordtxt+']}'
elif import_prof_schema == False:
    print('Schema and profiles not included in mdEditor file.')
    finaljson = '{\"data\": ['+recordtxt+','+prodrecordtxt+','+contacts_json+']}'

finaljson = json.loads(finaljson)
with open(os.path.join('initiated_metadata/', fn), 'w') as file: #Writes to new file in folder initiated_metadata of current directory
    try:
        file.write(json.dumps(finaljson, indent=4, sort_keys=False))
        print('mdEditor file successfully written to:\n    '+os.getcwd()+'\\initiated_metadata\\',fn,sep='')
    except:
        print('Error writing mdEditor file.')

Schema and profiles included in mdEditor file.
mdEditor file successfully written to:
    c:\Users\tpatterson\OneDrive - DOI\Documents\GitHub\DMPythonScript\ak_dmp_script_2026\initiated_metadata\fesmmm_028_Walrus_ Haulout_Satellite_Survey-init-mdeditor-20260805.json


***
# DMP to National SharePoint List
This final section of the script pushes the data management plan information into an Excel spreadsheet that can be ingested by the PowerAutomate Flow to push data to the National Data Management Plan SharePoint List (https://doimspp.sharepoint.com/sites/fws-data/Lists/Data%20Management%20Plans/AllItems.aspx).

If using this notebook only to initiated metadata, you may stop here.

## Data validation note
Because access to the SharePoint API was blocked by an administrator, data validation for SharePoint list values cannot occur within this notebook. All data validation will occur during the PowerAutomate Flow, and you will be emailed error messages.

It is particularly important to check the PartnerAgency list (https://doimspp.sharepoint.com/sites/fws-data/Lists/Agencies/AllItems.aspx?env=WebViewList) to verify the exact spelling/format of external partners in the data management plan. Either update the spelling/format in the DMP and re-run the script, or update the Excel spreadsheet after the row is appended, prior to running the PowerAutomate Flow.

## Data manipulation
The block of code below manipulates data from the project, product, and final_contacts pandas dataframes to fit the national DMP SharePoint list schema.

In [23]:
# Reformatting existing data to match the national form fields/data types

ntl_cols = ['ProjectTitle', 'ProjectDescription', 'CostCenter','FWSNationalProgram', 'FWSRegion', 'Keywords',
            'PartnerOrganization', 'DataSteward',  'DataTrustee', 'DataCustodianChoice', 'DataCustodianDOI',
            'DataCustodianNonDOI', 'DataProducerChoice', 'DataProducerDOI', 'DataProducerNonDOI',
            'ProjectTimeline', 'ProjectStartDate', 'ProjectEndDate', 'DataTypeFormatsCollected',
            'ExistingData', 'ProjectReferenceNumber', 'DataStorageLocation', 'DataStorageLinks',
            'DataAccess', 'DataAccessJustification', 'DataReviewSchedule', 'RequiredResources',
            'DataStandard', 'MetadataStandard', 'DataQualityAssurance', 'DataQualityControl',
            'RecordsSchedule', 'RecordsTypes', 'RecordsDisposition', 'Comments','Uploaded']

newentry = pd.DataFrame(columns=ntl_cols)

#Basic project info
newentry['ProjectTitle'] = project['projectTitle']
newentry['ProjectDescription'] = project['projectAbstract']

newentry['CostCenter'] = project[project.columns[
    pd.Series(project.columns).str.startswith('costCenter')]].values.flatten().tolist()[0]#.agg(';'.join, axis=1)
newentry['FWSNationalProgram'] = project[project.columns[
    pd.Series(project.columns).str.startswith('FWSProgram')]].values.flatten().tolist()[0].replace(
    'Office of Subsistence Management', 'Business Management and Operations')
newentry['FWSRegion'] = 'Region 7 Alaska'
newentry['Keywords'] = ['; '.join(map(str, l)) for l in project['projectKeywords']]

newentry['ProjectReferenceNumber'] = project['projectUIDList']
#Project timeline
if project['projectOngoing'][0] == 'True':
    newentry['ProjectTimeline'] = 'Ongoing'
else:
    newentry['ProjectTimeline'] = 'Planned Timeline'
    
startdate = project['projectStartDate'][0]
enddate = project['projectEndDate'][0]
newentry['ProjectStartDate'] = datetime.strptime(startdate, '%Y-%m-%d').date()
if enddate != '':
    newentry['ProjectEndDate'] = datetime.strptime(enddate, '%Y-%m-%d').date()
else:
    newentry['ProjectEndDate'] = None

if 'dataReviewSchedule' in project.columns:
    newentry['DataReviewSchedule'] = project['dataReviewSchedule']
else:
    newentry['DataReviewSchedule'] = 'Annual'


#Roles
trusteelist = final_contacts[(final_contacts.contact_type=='Trustee')]['Email'].tolist()
stewardlist = final_contacts[(final_contacts.contact_type=='Steward')]['Email'].tolist()
custodianlist = final_contacts[(final_contacts.contact_type=='Custodian') & (final_contacts.NonDOI=='False')]['Email'].tolist()
producerlist = final_contacts[(final_contacts.contact_type=='Originator') & (final_contacts.NonDOI=='False')]['Email'].tolist()

trusteeids = list(set(trusteelist))
newentry['DataTrustee'] = ';'.join(trusteeids)

stewardids = list(set(stewardlist))
newentry['DataSteward'] = ';'.join(stewardids)

custodianids = list(set(custodianlist))
if len(custodianids)>0:
    newentry['DataCustodianDOI'] = ';'.join(custodianids)
else:
    newentry['DataCustodianDOI'] = None
    
producerids = list(set(producerlist))
if len(producerids)>0:
    newentry['DataProducerDOI'] = ';'.join(producerids)
else:
    newentry['DataProducerDOI'] = None

if len(final_contacts[(final_contacts.contact_type=='Custodian') &(final_contacts.NonDOI=='True')])>0:
    newentry['DataCustodianNonDOI'] = '; '.join(list(set(final_contacts[(final_contacts.contact_type=='Custodian') &
                                                               (final_contacts.NonDOI=='True')
                                                              ][['FirstName','LastName','Email','Phone','Org'
                                                                ]].agg(' '.join, axis=1))))
else:
    newentry['DataCustodianNonDOI'] = None
    
if len(final_contacts[(final_contacts.contact_type=='Originator') & (final_contacts.NonDOI=='True')])>0:
    newentry['DataProducerNonDOI'] = '; '.join(list(set(final_contacts[(final_contacts.contact_type=='Originator') &
                                                       (final_contacts.NonDOI=='True')
                                                          ][['FirstName','LastName','Email','Org'
                                                            ]].agg(' '.join, axis=1))))
else: 
    newentry['DataProducerNonDOI'] = None

if newentry['DataCustodianNonDOI'][0]!=None and newentry['DataCustodianDOI'][0]!=None:
    newentry['DataCustodianChoice'] = 'Both'
elif newentry['DataCustodianNonDOI'][0]!=None and newentry['DataCustodianDOI'][0]==None:
    newentry['DataCustodianChoice'] = 'Non-DOI Person'
else:
    newentry['DataCustodianChoice'] = 'DOI Person'

    
if newentry['DataProducerNonDOI'][0]!=None and newentry['DataProducerDOI'][0]!=None:
    newentry['DataProducerChoice'] = 'Both'
elif newentry['DataProducerNonDOI'][0]!=None and newentry['DataProducerDOI'][0]==None:
    newentry['DataProducerChoice'] = 'Non-DOI Person'
else:
    newentry['DataProducerChoice'] = 'DOI Person'


#Existing data (source data)
existingdatalist = []
for i, row in products.iterrows():
    if row['productExists'] == 'True':
        val = ', '.join(row[['productTitle','productName','productURL']])
        existingdatalist.append(val)
newentry['ExistingData'] = '; '.join(existingdatalist)


#Data storage and repositories
storageloclist = []
storageurllist = []
if project['storageOneDrive'][0] == 'True':
    storageloclist.append('OneDrive')
    storageurllist.append(project['storageOneDriveURL'][0])
if project['storageTeams'][0] == 'True':
    storageloclist.append('Microsoft Teams')
    storageurllist.append(project['storageTeamsURL'][0])
if project['storageExternal'][0] == 'True':
    storageloclist.append('Local Storage')
if project['storageNetwork'][0] == 'True':
    storageloclist.append('Shared Drive')
    storageurllist.append(project['storageNetworkURL'][0])
    
repolinksdf = project[project.columns[pd.Series(project.columns).str.contains('repoURL')]].transpose().reset_index(drop=True)
repolinksdf.columns = ['value']
for i, row in repolinksdf.iterrows():
    storageurllist.append(row['value'])
    
reponames = str(project[project.columns[pd.Series(project.columns).str.contains('repoName')]].apply(lambda x: '; '.join(x), axis=1))
if 'ServCat' in reponames:
    storageloclist.append('ServCat')
if 'ArcGIS Online' in reponames:
    storageloclist.append('AGOL')
if 'InsideMaps' in reponames:
    storageloclist.append('FWS InsideMaps')
if 'GeoPlatform' in reponames:
    storageloclist.append('GeoPlatform')
if 'ScienceBase' in reponames:
    storageloclist.append('ScienceBase')
    
newentry['DataStorageLinks'] = '\n'.join(storageurllist)



#Records schedules
newentry['RecordsSchedule'] = project[project.columns[
    pd.Series(project.columns).str.startswith('recordsSchedule')]].agg('; '.join, axis=1)
newentry.loc[newentry['RecordsSchedule'] == '','RecordsSchedule'] = 'TBD'
newentry['RecordsTypes'] = project[project.columns[
    pd.Series(project.columns).str.startswith('recordsType')]].agg('; '.join, axis=1)
newentry.loc[newentry['RecordsTypes'] == '','RecordsTypes'] = 'TBD'
newentry['RecordsDisposition'] = project[project.columns[
    pd.Series(project.columns).str.startswith('recordsDisposition')]].agg('; '.join, axis=1)
newentry.loc[newentry['RecordsDisposition'] == '','RecordsDisposition'] = 'TBD'
if newentry['RecordsSchedule'].isin(['TBD']).any():
    print('Warning! Records schedules are missing and are required to submit the DMP. TBD has been used as a placeholder.')
if newentry['RecordsTypes'].isin(['TBD']).any():
    print('Warning! Records types are missing and are required to submit the DMP. TBD has been used as a placeholder.')
if newentry['RecordsDisposition'].isin(['TBD']).any():
    print('Warning! Records dispositions are missing and are required to submit the DMP. TBD has been used as a placeholder.')


#Partner information
partners = contacts_t[contacts_t.columns[
    pd.Series(contacts_t.columns).str.contains('Org')]].replace('', np.nan).values.flatten().tolist()
partners = [x for x in partners if str(x) != 'nan']
partnerids = list(set(partners))
if len(partnerids)>0:
    newentry['PartnerOrganization'] = ';'.join(partnerids)
else:
    newentry['PartnerOrganization'] = np.nan


#Data product info
prodtypelist = []
for i, row in products.iterrows():
    val = row['productResourceType'].replace('application','Work Flow/Code').replace('collection','Other').replace(
        'document','Other').replace('feature','Spatial').replace('map','Spatial').replace(
        'geographicDataset','Spatial').replace('photographicImage','Images').replace(
        'presentation','Audio Visual').replace('product','Other').replace('publication','Other').replace(
        'report','Other').replace('software','Work Flow/Code').replace('tabularDataset','Tabular')
    if val not in prodtypelist:
        prodtypelist.append(val)
prodtypelist = list(set(prodtypelist))
newentry['DataTypeFormatsCollected'] = ';'.join(prodtypelist)

products['productReqRes'] = products['productTitle']+': '+products['productResources']
newentry['RequiredResources'] = '; '.join(products[(products.productResources!='')&
                                                      (products.productResources!='NA')&
                                                      (products.productResources!='N/A')]['productReqRes'].unique())

products['productQA'] = products['productTitle']+': '+products['productQualityAssurance']
newentry['DataQualityAssurance'] = '; '.join(products[(products.productQualityAssurance!='')&
                                                      (products.productQualityAssurance!='NA')&
                                                      (products.productQualityAssurance!='N/A')]['productQA'])
products['productQC'] = products['productTitle']+': '+products['productQualityControl']
newentry['DataQualityControl'] = '; '.join(products[(products.productQualityControl!='')&
                                                      (products.productQualityControl!='NA')&
                                                      (products.productQualityControl!='N/A')]['productQC'])

newentry['DataStorageLocation'] = ';'.join(storageloclist)

newentry['DataAccess'] = ';'.join(products[['productRestriction']].drop_duplicates()['productRestriction'].str.replace(
    'open access','Open').replace('sensitive/protected','Sensitive/Protected').str.title().tolist())
newentry['DataAccessJustification'] = '; '.join(products[(products.productRestrictionJustification!='')
                                                        ]['productRestrictionJustification'].unique())


newentry['DataStandard'] = project[project.columns[
    pd.Series(project.columns).str.startswith('dataStandard')]].agg(';'.join, axis=1)
newentry.loc[newentry['DataStandard'] == '','DataStandard'] = 'Not Applicable'

mdstandardlist = project[project.columns[pd.Series(project.columns).str.startswith('metaStandardOtherType')]].values.flatten().tolist()
mdstandardlist = [x.replace('CSDGM (FDGC)','CSDGM (FGDC)') for x in mdstandardlist] #Fixing typo from early versions of data management plan


if project['metaStandardMdJSON'][0] == 'False':
        newentry.at[0,'MetadataStandard'] = ';'.join(mdstandardlist)
elif project['metaStandardMdJSON'][0] == 'True' and project['metaStandardOtherType'][0] == '':
    newentry['MetadataStandard'] = 'mdJSON'
else:
    mdstandardlist.append('mdJSON')
    newentry.at[0,'MetadataStandard'] = ';'.join(mdstandardlist)

newentry = newentry.replace(r'', None, regex=True)
newentry = newentry.replace(np.nan, None, regex=False)

newentry = newentry.replace('&amp;','&', regex=True
                           ).replace('&lt;','<', regex=True
                                    ).replace('&gt;','>', regex=True
                                             ).replace('&quot;','"', regex=True
                                                      ).replace('&#39;',"'", regex=True)
newentry['ProjectDescription'] = newentry['ProjectDescription'].str.replace('\\n',' ', regex=False)
newentry['DataQualityAssurance'] = newentry['DataQualityAssurance'].str.replace('\\n',' ', regex=False)
newentry['DataQualityControl'] = newentry['DataQualityControl'].str.replace('\\n',' ', regex=False)
newentry['RecordsSchedule'] = newentry['RecordsSchedule'].str.replace('schedule:*','schedule:*', regex=False)

display(newentry)

Warning! Records schedules are missing and are required to submit the DMP. TBD has been used as a placeholder.


C:\Users\tpatterson\AppData\Local\Temp\1\ipykernel_27908\572843762.py:168: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pd.Series(contacts_t.columns).str.contains('Org')]].replace('', np.nan).values.flatten().tolist()


,ProjectTitle,ProjectDescription,CostCenter,FWSNationalProgram,FWSRegion,Keywords,PartnerOrganization,DataSteward,DataTrustee,DataCustodianChoice,DataCustodianDOI,DataCustodianNonDOI,DataProducerChoice,DataProducerDOI,DataProducerNonDOI,ProjectTimeline,ProjectStartDate,ProjectEndDate,DataTypeFormatsCollected,ExistingData,ProjectReferenceNumber,DataStorageLocation,DataStorageLinks,DataAccess,DataAccessJustification,DataReviewSchedule,RequiredResources,DataStandard,MetadataStandard,DataQualityAssurance,DataQualityControl,RecordsSchedule,RecordsTypes,RecordsDisposition,Comments,Uploaded
0,Pacific Walrus Haulout Occupancy Survey: 2016-2025 Sentinel-1 & Sentinel-2 Satellite Imagery,"Pacific walrus (Odobenus rosmarus divergens) seasonally occupy coastal haulouts across their range, but seasonal space use patterns vary within and between years. Walruses are sensitive to human d...",FF07CAMM00,Ecological Services,Region 7 Alaska,Walrus; haulout; occupancy; satellite; abundance,None,devin_johnson@fws.gov,daniel_bjornlie@fws.gov,DOI Person,devin_johnson@fws.gov,None,DOI Person,devin_johnson@fws.gov,None,Ongoing,2026-02-01,2026-08-04,Other,None,None,ServCat,None,Open,None,Annual,"Pacific walrus haulout occupancy survey results - spreadsheet: NA, stored on shared drive and is <1TB.; Pacific Walrus Haulout Occupancy Survey - Report: NA, stored on shared drive and is <1TB.",Calendar Year,mdJSON,"Pacific walrus haulout occupancy survey results - spreadsheet: This product compiles observations from trained wildlife biologists in the Marine Mammal Management program, and each observation was...","Pacific walrus haulout occupancy survey results - spreadsheet: This product compiles observations from trained wildlife biologists in the Marine Mammal Management program, and each observation was...",TBD,"These records document USFWS scientific research and investigation of wildlife, wildlife habitat, fish health, fishery biology, fishery management, and scientific research and investigation of con...",a. Study Case Files. Retention: TEMPORARY. Destroy 10 years after study is completed.; b. Historical Study Case Files. Completed studies case files selected annually by the project director as per...,None,None


## Excel spreadsheet pull
The block of code below reads in the data from the existing National_DMP_Upload.xlsx file and appends the new record in a dataframe. It will also print a warning if a record already exists with the same title.

In [24]:
# Reading existing excel spreadsheet, to append new records to (xlsxwriter overrides the entire file, cannot append)
existing_xls = pd.read_excel(dmp_spreadsheet)
existing_xls['Uploaded'] = existing_xls['Uploaded'].fillna(False).astype(bool)

# Preparing data for Excel spreadsheet for Flow method

newentry_xls = newentry[ntl_cols]
newentry_xls['Uploaded'] = False

for i, row in newentry_xls.iterrows():
    val = row['ProjectTitle']
    if val in existing_xls['ProjectTitle'].tolist():
        print('!!! WARNING !!! A project with the title',val,'already exists in the Excel DMP document.\nPlease check for duplicates before proceeding.')

dat = pd.concat([existing_xls,newentry_xls]).reset_index(drop=True)
display(dat)

!!! WARNING !!! A project with the title Pacific Walrus Haulout Occupancy Survey: 2016-2025 Sentinel-1 & Sentinel-2 Satellite Imagery already exists in the Excel DMP document.
Please check for duplicates before proceeding.


,ProjectTitle,ProjectDescription,CostCenter,FWSNationalProgram,FWSRegion,Keywords,PartnerOrganization,DataTrustee,DataSteward,DataCustodianChoice,DataCustodianDOI,DataCustodianNonDOI,DataProducerChoice,DataProducerDOI,DataProducerNonDOI,ProjectTimeline,ProjectStartDate,ProjectEndDate,DataTypeFormatsCollected,ExistingData,ProjectReferenceNumber,DataStorageLocation,DataStorageLinks,DataAccess,DataAccessJustification,DataReviewSchedule,RequiredResources,DataStandard,MetadataStandard,DataQualityAssurance,DataQualityControl,RecordsSchedule,RecordsTypes,RecordsDisposition,Comments,Error,Uploaded
0,Pacific Walrus Haulout Occupancy Survey: 2016-2025 Sentinel-1 & Sentinel-2 Satellite Imagery,"Pacific walrus (Odobenus rosmarus divergens) seasonally occupy coastal haulouts across their range, but seasonal space use patterns vary within and between years. Walruses are sensitive to human d...",FF07CAMM00,Ecological Services,Region 7 Alaska,Walrus; haulout; occupancy; satellite; abundance,NaN,daniel_bjornlie@fws.gov,devin_johnson@fws.gov,DOI Person,devin_johnson@fws.gov,NaN,DOI Person,devin_johnson@fws.gov,NaN,Ongoing,2026-02-01 00:00:00,2026-08-04 00:00:00,Other,NaN,NaN,ServCat,NaN,Open,NaN,Annual,"Pacific walrus haulout occupancy survey results - spreadsheet: NA, stored on shared drive and is <1TB.; Pacific Walrus Haulout Occupancy Survey - Report: NA, stored on shared drive and is <1TB.",Calendar Year,mdJSON,"Pacific walrus haulout occupancy survey results - spreadsheet: This product compiles observations from trained wildlife biologists in the Marine Mammal Management program, and each observation was...","Pacific walrus haulout occupancy survey results - spreadsheet: This product compiles observations from trained wildlife biologists in the Marine Mammal Management program, and each observation was...",TBD,"These records document USFWS scientific research and investigation of wildlife, wildlife habitat, fish health, fishery biology, fishery management, and scientific research and investigation of con...",a. Study Case Files. Retention: TEMPORARY. Destroy 10 years after study is completed.; b. Historical Study Case Files. Completed studies case files selected annually by the project director as per...,NaN,NaN,False
1,Pacific Walrus Haulout Occupancy Survey: 2016-2025 Sentinel-1 & Sentinel-2 Satellite Imagery,"Pacific walrus (Odobenus rosmarus divergens) seasonally occupy coastal haulouts across their range, but seasonal space use patterns vary within and between years. Walruses are sensitive to human d...",FF07CAMM00,Ecological Services,Region 7 Alaska,Walrus; haulout; occupancy; satellite; abundance,None,daniel_bjornlie@fws.gov,devin_johnson@fws.gov,DOI Person,devin_johnson@fws.gov,None,DOI Person,devin_johnson@fws.gov,None,Ongoing,2026-02-01,2026-08-04,Other,None,None,ServCat,None,Open,None,Annual,"Pacific walrus haulout occupancy survey results - spreadsheet: NA, stored on shared drive and is <1TB.; Pacific Walrus Haulout Occupancy Survey - Report: NA, stored on shared drive and is <1TB.",Calendar Year,mdJSON,"Pacific walrus haulout occupancy survey results - spreadsheet: This product compiles observations from trained wildlife biologists in the Marine Mammal Management program, and each observation was...","Pacific walrus haulout occupancy survey results - spreadsheet: This product compiles observations from trained wildlife biologists in the Marine Mammal Management program, and each observation was...",TBD,"These records document USFWS scientific research and investigation of wildlife, wildlife habitat, fish health, fishery biology, fishery management, and scientific research and investigation of con...",a. Study Case Files. Retention: TEMPORARY. Destroy 10 years after study is completed.; b. Historical Study Case Files. Completed studies case files selected annually by the project director as per...,None,NaN,False


## Writing to Excel
This final block of code writes the dataframe displayed above to the National_DMP_Upload.xlsx file. The overrides existing data in the spreadsheet, so it is important to have this file in One Drive for back-ups in case the script errors out and only partially inputs data.

In [25]:
#  Overwriting existing XLSX DMP file with new entry included

# libraries for writing to national dmp list
import traceback
import xlsxwriter
import string
import openpyxl

writer = None

try:
    wide_cols = ['ProjectDescription','RequiredResources','DataQualityAssurance','DataQualityControl',
                'RecordsSchedule','RecordsTypes','RecordsDisposition']
    narrow_cols = dat.columns.difference(wide_cols, sort=False)
    alpha = list(string.ascii_uppercase) + [letter1+letter2 for letter1 in string.ascii_uppercase for letter2 in string.ascii_uppercase]
    d = dict(zip(range(len(dat.columns)), alpha))

    with pd.ExcelWriter(dmp_spreadsheet, engine='xlsxwriter') as writer:
        dat.to_excel(writer, sheet_name='DMP', index=False)
    
        worksheet = writer.sheets['DMP']
        workbook = writer.book
    
        wrap_format = workbook.add_format({'text_wrap': True})
    
        for col in dat.columns.get_indexer(wide_cols):
            excel_header  =  d[col] + ':' + d[col]
            worksheet.set_column(excel_header, 50, wrap_format)
            
        for col in dat.columns.get_indexer(narrow_cols):
            excel_header  =  d[col] + ':' + d[col]
            worksheet.set_column(excel_header, 20, wrap_format)
    
        header_cell_format = workbook.add_format()
        header_cell_format.set_align('left')
        header_cell_format.set_align('vcenter')
    
        col_names = [{'header': col_name} for col_name in dat.columns]
        worksheet.add_table(0, 0, dat.shape[0], dat.shape[1]-1, {
            'columns': col_names,
            'style': 'Table Style Medium 1',
            'name': 'National_DMPs'
        })
    
        for i, col in enumerate(col_names):
            worksheet.write(0, i, col['header'], header_cell_format)

    print('Excel spreadsheet successfully overwritten.')

except Exception:
    traceback.print_exc()
    print('There was an error writing the spreadsheet.\nIf you cannot resolve this error, please restore your file to the previous version on OneDrive.')

Excel spreadsheet successfully overwritten.
